# Reading Between the Lines
## How Much Information Remains After Fitting a Stellar Spectrum?

Author: Sven Buder (ANU), David Hogg (NYU)  
GitHub repository: https://github.com/svenbuder/residual_cannon  
arXiv: YYMM.NNNNN

In [ ]:
# Setting a new & isolated conda environment for this project in terminal (note specific numpy version):
# conda create -n py_cannon python=3.12
# conda activate py_cannon
# pip install matplotlib astropy
# pip install "numpy==2.2.*"
# pip install git+https://github.com/ANU-RSAA/AnniesLasso.git

In [ ]:
# Importing packages and software version reporting
try:
    %matplotlib inline
    %config InlineBackend.figure_format='retina'
except:
    pass

import sys, warnings; print("python version:", sys.version)

import numpy as np; print('numpy version: ',np.__version__)

import scipy; print('scipy version: ',scipy.__version__)
from scipy.signal import find_peaks

import IPython; print('IPython version: ',IPython.__version__)

import matplotlib; print('matplotlib version: ',matplotlib.__version__)
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MaxNLocator

import astropy; print('astropy version: ',astropy.__version__)
from astropy.table import Table, join
from astropy.io import fits
import astropy.units as u
from astropy.units import UnitsWarning

import thecannon as tc; print('thecannon version: ',tc.__version__)

# Reporting software versions to a .tex file for inclusion in the paper
np.savetxt('tex_text/software_info.tex',
              [f"\\textsc{{python}} (version {sys.version.split()[0]}) and included its packages",
                f"\\textsc{{astropy}} \\citep[v. {astropy.__version__};]{{Robitaille2013,PriceWhelan2018}},",
                f"\\textsc{{IPython}} \\citep[v. {IPython.__version__};]{{ipython}},",
                f"\\textsc{{matplotlib}} \\citep[v. {matplotlib.__version__};]{{matplotlib}},",
                f"\\textsc{{NumPy}} \\citep[v. {np.__version__};]{{numpy}},",
                f"\\textsc{{scipy}} \\citep[v. {scipy.__version__};]{{Scipy}};",
                f"\\textsc{{topcat}} \\citep[v. 4.7;]{{Taylor2005}};",
                f"\\textsc{{thecannon}} \\citep[v. {tc.__version__} from \\url{{https://github.com/ANU-RSAA/AnniesLasso}} maintained by Marc White from ANU RSAA/AITC's Software Group]{{Ness2015, Casey2016}}."],
              fmt="%s"
)
print('Saved software versions to tex_text/software_info.tex')

In [ ]:
# rerun_everything = True
rerun_everything = False

# 2 Data
## 2.1 GALAH DR4 spectra and labels

In [ ]:
# Define common wavelength grid for Cannon training and testing
common_wavelength = np.concatenate((
    np.arange(4719.0299,4893.8808,0.0460),
    np.arange(5655.5298,5863.0182,0.0547),
    np.arange(6486.0104,6726.4572,0.0631),
    np.arange(7689.1261,7874.3127,0.0735)
))

In [ ]:
example_spectrum = Table.read('spectra/170710/170710003201361/170710003201361_allstar_fit_spectrum.fits')

# Transfer mask from example spectrum onto common wavelength grid (most consistently with float interpolation and cut at 0.8)
mask_common_wavelength = np.interp(common_wavelength, example_spectrum['wave'], np.array(example_spectrum['mob'], dtype=float))
mask_common_wavelength[mask_common_wavelength < 0.8] = 0.0
mask_common_wavelength[mask_common_wavelength >= 0.8] = 1.0
mask_common_wavelength = mask_common_wavelength.astype(bool)
print(f"Number of pixels in common wavelength grid: {len(common_wavelength)}")
print(f"Number of pixels in common wavelength grid that are masked: {len(np.where(mask_common_wavelength==False)[0])}")
print(f"Number of pixels in common wavelength grid that are unmasked: {len(np.where(mask_common_wavelength==True)[0])}")
print(f"Percentage of pixels in common wavelength grid that are masked: {100*len(np.where(mask_common_wavelength==False)[0])/len(common_wavelength):.1f}")

In [ ]:
# Extract skylines and telluric lines used by reduction pipeline for this object
reduction_pipeline_wave = []
reduction_pipeline_sob = []
reduction_pipeline_uob = []
reduction_pipeline_sky = []
reduction_pipeline_telluric = []

# loop over the 4 FITS files of the reduction
for ccd in [1,2,3,4]:
    fits_file = fits.open('data/galah_dr4_iraf_6p1_170710003201361'+str(ccd)+'.fits')
    # wavelength
    reduction_pipeline_wave.append(fits_file[3].header['CRVAL1'] + np.arange(fits_file[3].header['NAXIS1'])*fits_file[3].header['CDELT1'])
    # 1 == stellar normalised
    reduction_pipeline_sob.append(fits_file[1].data)
    # 2 == stellar normalised uncertainty
    reduction_pipeline_uob.append(fits_file[1].data * fits_file[2].data)
    # 3 == sky
    reduction_pipeline_sky.append(fits_file[3].data)
    # 4 == telluric
    reduction_pipeline_telluric.append(fits_file[4].data)

wave_sky_telluric = np.concatenate(reduction_pipeline_wave) * (1 - fits_file[3].header['BARYEFF']/299792.458)
reduction_pipeline_sob = np.concatenate(reduction_pipeline_sob)
reduction_pipeline_uob = np.concatenate(reduction_pipeline_uob)
reduction_pipeline_sky = np.concatenate(reduction_pipeline_sky)
reduction_pipeline_sky /= np.median(reduction_pipeline_sky) # normalise sky to median of 1 across the full CCD
reduction_pipeline_telluric = np.concatenate(reduction_pipeline_telluric)

reduction_sky_tellurics = Table()
reduction_sky_tellurics['wave'] = common_wavelength
reduction_sky_tellurics['wave_sky'] = common_wavelength
reduction_sky_tellurics['sob'] = np.interp(common_wavelength, wave_sky_telluric, reduction_pipeline_sob)
reduction_sky_tellurics['uob'] = np.interp(common_wavelength, wave_sky_telluric, reduction_pipeline_uob)
reduction_sky_tellurics['sky'] = np.interp(common_wavelength, wave_sky_telluric, reduction_pipeline_sky)
reduction_sky_tellurics['telluric'] = np.interp(common_wavelength, wave_sky_telluric, reduction_pipeline_telluric)
reduction_sky_tellurics.write('data/reduction_sky_tellurics.fits', overwrite=True)
print('Saved reduction sky and telluric information of 170710003201361 on original wavelength (1-BARYEFF/299792.458) grid to data/reduction_sky_tellurics.fits')

In [ ]:
# Read in example spectrum and calculate sigma-clipped residuals as well as chi2 mask
if rerun_everything:

    # minimum cut
    bad_flux = example_spectrum['sob'] > 1.05
    example_spectrum['sob'][bad_flux] = example_spectrum['smod'][bad_flux]

    # significance
    sigma_1_pos = (example_spectrum['sob'] - example_spectrum['smod'])/example_spectrum['uob'] > 1
    sigma_1_neg = (example_spectrum['sob'] - example_spectrum['smod'])/example_spectrum['uob'] < -1
    sigma_3_pos = (example_spectrum['sob'] - example_spectrum['smod'])/example_spectrum['uob'] > 3
    sigma_3_neg = (example_spectrum['sob'] - example_spectrum['smod'])/example_spectrum['uob'] < -3

    # clipped residuals
    residuals_1 = example_spectrum['sob'] - example_spectrum['smod']
    residuals_1[sigma_1_pos] = example_spectrum['uob'][sigma_1_pos]
    residuals_1[sigma_1_neg] = -example_spectrum['uob'][sigma_1_neg]
    residuals_3 = example_spectrum['sob'] - example_spectrum['smod']
    residuals_3[sigma_3_pos] = 3*example_spectrum['uob'][sigma_3_pos]
    residuals_3[sigma_3_neg] = -3*example_spectrum['uob'][sigma_3_neg]

    mask_start_end = []

    found_start = False
    found_end = False

    for index in range(len(example_spectrum['wave'])-1):
        if found_start == False:
            if example_spectrum['mob'][index] == False:
                start = index
                found_start = True
        if found_start == True and found_end == False:
            if example_spectrum['mob'][index] == True:
                end = index
                found_end = True
        if found_start == True and found_end == True:
            mask_start_end.append((example_spectrum['wave'][start], example_spectrum['wave'][end]))
            found_start = False
            found_end = False

In [ ]:
# Example Spectrum Plot

if rerun_everything:
    f, gs = plt.subplots(4,1,figsize=(5,7),sharey=True)

    mask_used = False
    for ccd in [1, 2, 3, 4]:

        ax = gs[ccd-1]
        in_ccd = ((example_spectrum["wave"] > (3 + ccd) * 1000) & (example_spectrum["wave"] < (4 + ccd) * 1000))

        ax.plot(example_spectrum['wave'][in_ccd], example_spectrum['sob'][in_ccd], 'k', label='Data', lw = 0.75)
        ax.plot(example_spectrum['wave'][in_ccd], example_spectrum['smod'][in_ccd], 'C0', label='Model', lw = 0.5)

        colors = ['navajowhite', 'orange', 'darkred']

        ax.fill_between(example_spectrum['wave'][in_ccd], 
            np.zeros(np.shape(example_spectrum['sob'][in_ccd])[0]), 
            (example_spectrum['sob'][in_ccd] - example_spectrum['smod'][in_ccd]),
            color = colors[2],label=r'$\Delta f > 3\sigma$',lw=0.75
        )
        ax.fill_between(example_spectrum['wave'][in_ccd], 
            np.zeros(np.shape(example_spectrum['sob'][in_ccd])[0]), 
            residuals_3[in_ccd],
            color = colors[1],label=r'$\Delta f > 2\sigma$',lw=0.75
        )
        ax.fill_between(example_spectrum['wave'][in_ccd], 
            np.zeros(np.shape(example_spectrum['sob'][in_ccd])[0]), 
            residuals_1[in_ccd],
            color = colors[0],label=r'$\Delta f \leq 1\sigma$',lw=0.75
        )
        
        for mask in mask_start_end:
            if (mask[0] > (3 + ccd) * 1000) & (mask[1] < (4 + ccd) * 1000):
                
                if mask_used:
                    legend = '_nolegend_'
                else:
                    mask_used = True
                    legend = 'DR4 Masked'
                ax.axvspan(mask[0], mask[1], ymin = 0.0, ymax=0.1, color='lightgrey', lw = 0.75, label = legend)#, color='lightgrey', alpha=0.5, label='masked region')
        
        if ccd == 1:
            ax.legend(ncol=3,fontsize=12, loc='upper right', bbox_to_anchor=(1.02, 1.55), frameon=False)
        if ccd == 4:
            ax.set_xlabel(r'Wavelength $\lambda_\mathrm{air}~/~\mathrm{\AA}$', fontsize=12)
        ax.set_ylabel(r'Flux $f_\lambda~/~\mathrm{norm.}$', fontsize=12)

    plt.tight_layout(w_pad=0.0, h_pad=0.0)
    plt.savefig('figures/example_spectrum_170710003201361.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

## 2.2 Sample Selection

In [ ]:
# Read in GALAH DR4 and down-size to 170710 - 170712

if rerun_everything:
    dr4 = Table.read('data/galah_dr4_allstar_240705.fits')
    dr4_tmass_ids = (dr4[['tmass_id']]).copy()
    dr4['date'] = np.array([str(dr4['sobject_id'][x])[:6] for x in range(len(dr4['sobject_id']))])
    dr4 = dr4[(dr4['date'] >= '170710') & (dr4['date'] <= '170712')]
    print(len(dr4), 'stars in 170710 - 170712')

### 2.2.1 Baseline Sample

In [ ]:
# Load or select&save baseline sample for Cannon training and testing

if rerun_everything:
    baseline_sample = (
        (dr4['flag_sp'] < 2**14) &
        (dr4['chi2_sp'] < 2) &
        (dr4['snr_px_ccd1'] > 10) & (dr4['snr_px_ccd2'] > 10) & (dr4['snr_px_ccd3'] > 10) & (dr4['snr_px_ccd4'] > 10) &
        np.isfinite(dr4['ebv']) & np.isfinite(dr4['a_ks'])
    )
    print(len(dr4[baseline_sample]), 'stars in baseline sample')

    # plot and print SNR distribution of baseline sample
    f, ax = plt.subplots(1,1)
    ccd_colors = ['C0','C2','C3','k']
    for ccd in [1,2,3,4]:
        ax.hist(dr4['snr_px_ccd'+str(ccd)][baseline_sample], bins=np.linspace(0,200,21), histtype='step', label='CCD '+str(ccd), color=ccd_colors[ccd-1])
        # print SNR percentiles
        print(f"Median SNR for CCD {ccd}: {np.percentile(dr4['snr_px_ccd'+str(ccd)][baseline_sample], 50):.0f}")
    ax.set_xlabel('SNR per pixel')
    ax.set_ylabel('Number of stars')
    ax.legend()
    plt.tight_layout()
    plt.show()
    plt.close()

    # fast rotators and hot stars (to show as excluded in abundance sample)
    fast_rotators_and_hot_stars = dr4[(
        baseline_sample &
        (
            (dr4['vsini'] > 15) | (dr4['teff'] > 6500)
        )
    )]

    baseline_residual_flux = []
    baseline_residual_ivar = []
    baseline_flux = []

    # Interpolate onto same wavelength
    for sobject_id in dr4['sobject_id'][baseline_sample]:

        spectrum = Table.read('spectra/'+str(sobject_id)[:6]+'/'+str(sobject_id)+'/'+str(sobject_id)+'_allstar_fit_spectrum.fits')

        baseline_residual_flux.append(np.interp(common_wavelength,spectrum['wave'],spectrum['sob']-spectrum['smod']))
        baseline_residual_ivar.append(1./(np.interp(common_wavelength,spectrum['wave'],spectrum['uob']))**2)
        baseline_flux.append(np.interp(common_wavelength,spectrum['wave'],spectrum['sob']))

    baseline_residual_flux = np.array(baseline_residual_flux)
    baseline_residual_ivar = np.array(baseline_residual_ivar)
    baseline_flux = np.array(baseline_flux)

    np.savez('data/flux_ivar_labels_model1.npz', flux=baseline_residual_flux, ivar=baseline_residual_ivar, labels=dr4[['sobject_id','teff','logg','fe_h']][baseline_sample])
    np.savez('data/flux_ivar_labels_model3.npz', flux=baseline_residual_flux, ivar=baseline_residual_ivar, labels=dr4[['sobject_id','teff','logg','fe_h','a_ks']][baseline_sample])

    np.savez('data/flux_percentiles_baseline.npz', flux_percentiles=np.percentile(baseline_flux, [5,16,50,84,95], axis=0))

flux_percentiles = np.load('data/flux_percentiles_baseline.npz')['flux_percentiles']

model1_data = np.load('data/flux_ivar_labels_model1.npz')
print('Loaded (residual) flux, ivar, and labels for model 1 with '+str(np.shape(model1_data['flux']))+' spectra and labels '+str(list(model1_data['labels'].dtype.names)))

model3_data = np.load('data/flux_ivar_labels_model3.npz')
print('Loaded (residual) flux, ivar, and labels for model 3 with '+str(np.shape(model3_data['flux']))+' spectra and labels '+str(list(model3_data['labels'].dtype.names)))

In [ ]:
# Give minimum and maximum as well as median and 16-84th percentile ranges of the labels in the baseline sample
print('Label ranges in baseline sample:')
for label in model3_data['labels'].dtype.names[1:]:
    print(f"{label}: {np.min(model3_data['labels'][label]):.2f} - {np.max(model3_data['labels'][label]):.2f} (median: {np.median(model3_data['labels'][label]):.2f}, 16-84th percentile: {np.percentile(model3_data['labels'][label], 16):.2f} - {np.percentile(model3_data['labels'][label], 84):.2f})")

In [ ]:
# Show Teff-logg diagram of the baseline sample

if rerun_everything:
    f, ax = plt.subplots(figsize=(6,4))
    s = ax.scatter(
        model1_data['labels']['teff'],
        model1_data['labels']['logg'],
        c = model1_data['labels']['fe_h'],
        cmap = 'RdYlBu',
        s = 4,
        vmin = -1.5, vmax = 0.5
    )
    ax.set_xlabel(r'$T_\mathrm{eff}~/~\mathrm{K}$',fontsize=15)
    ax.set_ylabel(r'$\log (g~/~\mathrm{cm\,s^{-2}})$',fontsize=15)
    ax.set_xlim(ax.get_xlim()[::-1])
    ax.set_ylim(ax.get_ylim()[::-1])
    ax.text(0.05,0.95,'Models 1 & 3: ' + str(len(model1_data['labels']['sobject_id'])) + ' Stars',va='top',ha='left',transform=ax.transAxes,fontsize=15)
    cbar = plt.colorbar(s,ax=ax)
    cbar.set_label('[Fe/H]',fontsize=15)

    ax.set_xlim(8250, 3500)
    ax.set_ylim(5.1,0.4)

    plt.tight_layout()
    plt.savefig('figures/labels_m1_teff_logg_feh.png',dpi=200,bbox_inches='tight')
    plt.show()
    plt.close()

### 2.2.2 Abundance Sample

In [ ]:
# Load or select&save abundance sample for Cannon training and testing

if rerun_everything:
    abundance_sample = (
        baseline_sample &
        (dr4['snr_px_ccd1'] > 25) & (dr4['snr_px_ccd2'] > 25) & (dr4['snr_px_ccd3'] > 25) & (dr4['snr_px_ccd4'] > 25) &
        (dr4['vsini'] < 15) &
        (dr4['teff'] < 6500) &
        (dr4['flag_mg_fe'] == 0) & # alpha-proces
        (dr4['flag_na_fe'] == 0) & # odd-Z; more measurements than Al
        (dr4['flag_mn_fe'] == 0) & # iron-peak
        (dr4['flag_ba_fe'] == 0) # s-process; more measurements than Y
    )
    print(len(dr4[abundance_sample]), 'stars in abundance sample')

    abundance_residual_flux = []
    abundance_residual_ivar = []

    # Interpolate onto same wavelength
    for sobject_id in dr4['sobject_id'][abundance_sample]:
        
        spectrum = Table.read('spectra/'+str(sobject_id)[:6]+'/'+str(sobject_id)+'/'+str(sobject_id)+'_allstar_fit_spectrum.fits')

        abundance_residual_flux.append(np.interp(common_wavelength,spectrum['wave'],spectrum['sob']-spectrum['smod']))
        abundance_residual_ivar.append(1./(np.interp(common_wavelength,spectrum['wave'],spectrum['uob']))**2)

    abundance_residual_flux = np.array(abundance_residual_flux)
    abundance_residual_ivar = np.array(abundance_residual_ivar)

    np.savez('data/flux_ivar_labels_model2.npz', flux=abundance_residual_flux, ivar=abundance_residual_ivar, labels=dr4[['sobject_id','teff','logg','fe_h','na_fe','mg_fe','mn_fe','ba_fe']][abundance_sample])

model2_data = np.load('data/flux_ivar_labels_model2.npz')
print('Loaded flux, ivar, and labels for model 2 with '+str(np.shape(model2_data['flux']))+' spectra and labels '+str(list(model2_data['labels'].dtype.names)))

In [ ]:
# Show Teff-logg diagram of the abundance sample

if rerun_everything:
    f, ax = plt.subplots(figsize=(6,4))

    # plot neglected fast rotators and hot stars
    s = ax.scatter(
        fast_rotators_and_hot_stars['teff'],
        fast_rotators_and_hot_stars['logg'],
        marker='x',
        s=3,
        linewidths=0.5,
        alpha = 0.5,
        color = 'grey',
        zorder = 1
    )
    s = ax.scatter(
        model2_data['labels']['teff'],
        model2_data['labels']['logg'],
        c = model2_data['labels']['fe_h'],
        cmap = 'RdYlBu',
        s = 4,
        vmin = -1.5, vmax = 0.5,
        zorder = 2
    )
    ax.set_xlabel(r'$T_\mathrm{eff}~/~\mathrm{K}$',fontsize=15)
    ax.set_ylabel(r'$\log (g~/~\mathrm{cm\,s^{-2}})$',fontsize=15)
    ax.set_xlim(ax.get_xlim()[::-1])
    ax.set_ylim(ax.get_ylim()[::-1])
    ax.text(0.05,0.95,'Model 2: ' + str(len(model2_data['labels']['sobject_id'])) + ' Stars',va='top',ha='left',transform=ax.transAxes,fontsize=15)
    cbar = plt.colorbar(s,ax=ax)
    cbar.set_label('[Fe/H]',fontsize=15)

    ax.text(8000,4.4,'Excluded\nHot Stars and Fast Rotators',ha='left',va='top',fontsize=8,color='grey')

    ax.set_xlim(8250, 3500)
    ax.set_ylim(5.1,0.4)

    # inset with cumulative distribution oftypical absolute residuals
    axins = ax.inset_axes([0.175, 0.5, 0.35, 0.35])
    axins.set_xlabel('[Fe/H]',fontsize=10)
    axins.set_ylabel('[Mg/Fe]',fontsize=10)

    feh_lim = (-2.35,0.65)
    xfe_lim = (-0.2,0.65)
    axins.set_xlim(feh_lim)
    axins.set_ylim(xfe_lim)

    axins.scatter(
        model2_data['labels']['fe_h'],
        model2_data['labels']['mg_fe'],
        c = 'k',
        s = 2,
        lw = 0.
    )
    axins.hist2d(
        model2_data['labels']['fe_h'],
        model2_data['labels']['mg_fe'],
        bins = (np.linspace(feh_lim[0], feh_lim[1], 50), np.linspace(xfe_lim[0], xfe_lim[1], 50)),
        cmap = 'Greys_r',
        cmin = 5,
        norm = LogNorm()
    )
    axins.tick_params(axis='both', labelsize=9)
    axins.set_xticks([-2, -1, 0])
    axins.set_yticks([0, 0.25, 0.5])

    plt.tight_layout()
    plt.savefig('figures/labels_m2_teff_logg_feh_xfe.png',dpi=200,bbox_inches='tight')
    plt.show()
    plt.close()

### 2.2.3 Activity Sample

In [ ]:
# Crossmatch with activity studies by Boro Saikia+ 2018, Brown+ 2022, Gomes da Silva+ 2021, Lorenzo Oliveira+ 2018, and Zhang+ 2024

if rerun_everything:
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', UnitsWarning)

        # We have x-matched the following catalogues with 2MASS via 5" RA/DEC:
        rhk_literature = dict()
        rhk_literature['BoroSaikia2018_AandA_616_108'] = Table.read('data/BoroSaikia2018_AandA_616_108_2MASS.fits')
        rhk_literature['Brown2022_MNRAS_514_4300'] = Table.read('data/Brown2022_MNRAS_514_4300_2MASS.fits')
        rhk_literature['GomesDaSilva_2021_AandA_646_77'] = Table.read('data/GomesDaSilva_2021_AandA_646_77_2MASS.fits')
        rhk_literature['LorenzoOliveira_2018_AandA_619_73'] = Table.read('data/LorenzoOliveira_2018_AandA_619_73_2MASS.fits')
        rhk_literature['Zhang_2024_AandA_688_23'] = Table.read('data/Zhang_2024_AandA_688_23_GALAH_DR4.fits')

        # Now let's crossmatch with GALAH DR4 via 2MASS IDs
        for rhk_source in rhk_literature.keys():
            try:
                rhk_literature[rhk_source].rename_column('2MASS','tmass_id')
            except:
                pass

            dr4_rhk = join(rhk_literature[rhk_source], dr4_tmass_ids, keys='tmass_id',metadata_conflicts='silent')
            print('Overlap with stellar activity study by '+rhk_source+':',len(dr4_rhk))


In [ ]:
# Load or select&save activity sample (baseline sample x Zhang+ 2024) for Cannon training and testing

if rerun_everything:

    Zhang_2024_AandA_688_23 = Table.read('data/Zhang_2024_AandA_688_23_GALAH_DR4.fits')
    Zhang_2024_AandA_688_23.rename_column('logRHKcl', 'logRHK')

    activity_sample = join(dr4[baseline_sample], Zhang_2024_AandA_688_23, keys = 'sobject_id', metadata_conflicts='silent')

    # Get rid of the 1 star with significantly lower logg = 3.4
    activity_sample = activity_sample[activity_sample['logg'] > 3.8]

    print(len(activity_sample), 'stars in activity sample')

    activity_residual_flux = []
    activity_residual_ivar = []

    # Interpolate onto same wavelength
    for sobject_id in activity_sample['sobject_id']:
        
        spectrum = Table.read('spectra/'+str(sobject_id)[:6]+'/'+str(sobject_id)+'/'+str(sobject_id)+'_allstar_fit_spectrum.fits')

        activity_residual_flux.append(np.interp(common_wavelength,spectrum['wave'],spectrum['sob']-spectrum['smod']))
        activity_residual_ivar.append(1./(np.interp(common_wavelength,spectrum['wave'],spectrum['uob']))**2)

    activity_residual_flux = np.array(activity_residual_flux)
    activity_residual_ivar = np.array(activity_residual_ivar)

    np.savez('data/flux_ivar_labels_model4.npz', flux=activity_residual_flux, ivar=activity_residual_ivar, labels=activity_sample[['sobject_id','teff','logg','fe_h','logRHK']])

model4_data = np.load('data/flux_ivar_labels_model4.npz')
print('Loaded flux, ivar, and labels for model 4 with '+str(np.shape(model4_data['flux']))+' spectra and labels '+str(list(model4_data['labels'].dtype.names)))

In [ ]:
# Show Teff-logg diagram of the activity sample

if rerun_everything:
    f, ax = plt.subplots(figsize=(6,4))
    s = ax.scatter(
        model4_data['labels']['teff'],
        model4_data['labels']['logg'],
        c = model4_data['labels']['logRHK'],
        cmap = 'RdYlBu',
        s = 4
    )
    ax.set_xlabel(r'$T_\mathrm{eff}~/~\mathrm{K}$',fontsize=15)
    ax.set_ylabel(r'$\log (g~/~\mathrm{cm\,s^{-2}})$',fontsize=15)
    ax.set_xlim(ax.get_xlim()[::-1])
    ax.set_ylim(ax.get_ylim()[::-1])
    ax.text(0.05,0.95,'Model 4: ' + str(len(model4_data['labels']['sobject_id'])) + ' Stars',va='top',ha='left',transform=ax.transAxes,fontsize=15)
    cbar = plt.colorbar(s,ax=ax)
    cbar.set_label(r"$\log R'_\mathrm{HK}$",fontsize=15)

    axins = ax.inset_axes([0.66, 0.66, 0.33, 0.33])
    axins.hist(activity_sample['logRHK'], bins=np.linspace(-5.,-3.9,16), histtype="step")
    axins.set_xlabel(r"$\log R'_\mathrm{HK}$", fontsize=12)
    axins.set_ylabel("Nr", fontsize=12)

    ax.set_xlim(8250, 3500)
    ax.set_ylim(5.1,0.4)

    plt.tight_layout()
    plt.savefig('figures/labels_m4_teff_logg_logrhk.png',dpi=200,bbox_inches='tight')
    plt.show()
    plt.close()

# 3 Analysis

## 3.1 A systematic audit of spectrum residuals

### 3.1.1 How significant are the residuals and where?

In [ ]:
# Calculate absolute residual flux percentiles & their significance for model 1 & how often residuals are above/below +/- a chosen sigma
residual_flux_percentiles = np.percentile(model1_data['flux'], q=[16,50,84], axis=0)
absolute_residual_flux_percentiles = np.percentile(np.abs(model1_data['flux']), q=[16,50,84], axis=0)
significance_residual_flux_percentiles = np.percentile(np.abs(model1_data['flux']) * np.sqrt(model1_data['ivar']), q=[16,50,84], axis=0)

In [ ]:
residual_flux_significance = model1_data['flux'] * np.sqrt(model1_data['ivar'])

# We expect 68% to be within +/- 1 sigma and 99.7% to be within +/- 3 sigma for a normal distribution. Let's check
total_pixels = np.prod(model1_data['flux'].shape)
print('Total number of pixels across all spectra:', f"{total_pixels/1e6:.0f} million")

# And actual numbers?
within_1sigma = np.sum((residual_flux_significance > -1) & (residual_flux_significance < 1))
within_3sigma = np.sum((residual_flux_significance > -3) & (residual_flux_significance < 3))
print(f"Percentage of pixels with residual flux significance within +/- 1 sigma: {100*within_1sigma/total_pixels:.1f}%")
print(f"Percentage of pixels with residual flux significance within +/- 3 sigma: {100*within_3sigma/total_pixels:.1f}%")

# How many more outliers than expected do we have for 3 sigma?
expected_outliers_3sigma = total_pixels * (1 - 0.997) # 0.3% expected to be outside of +/- 3 sigma
actual_outliers_3sigma = total_pixels - within_3sigma
print(f"Expected number of pixels with residual flux significance outside of +/- 3 sigma: {expected_outliers_3sigma/1e6:.1f} million")
print(f"Actual number of pixels with residual flux significance outside of +/- 3 sigma: {actual_outliers_3sigma/1e6:.1f} million")

In [ ]:
# Some of them are neighbouring, so how many are there if we group neighbouring pixels together?
residuals_below_minus_0_1_grouped = find_peaks(np.round(-residual_flux_percentiles[1],2), height=0.1)[0]
print(f"Number of groups of neighbouring pixels with median residual flux below -0.1: {len(residuals_below_minus_0_1_grouped)}")

# And what wavelengths are these peaks at?
print("Wavelengths of pixels with median residual flux below -0.1:")
for index in residuals_below_minus_0_1_grouped:
    print(f"{common_wavelength[index]:.2f} Å")

# Some of them are neighbouring, so how many are there if we group neighbouring pixels together?
residuals_below_0_1_grouped = find_peaks(np.round(residual_flux_percentiles[1],2), height=0.1)[0]
print(f"Number of groups of neighbouring pixels with median residual flux above 0.1: {len(residuals_below_0_1_grouped)}")

# And what wavelengths are these peaks at?
print("Wavelengths of pixels with median residual flux above 0.1:")
for index in residuals_below_0_1_grouped:
    print(f"{common_wavelength[index]:.2f} Å")

In [ ]:
# Plot absolute residual flux percentiles & their significance for model 1

if rerun_everything:
    f, gs = plt.subplots(1,4,figsize=(15,3), sharey=True)
    for ccd in [1, 2, 3, 4]:
        
        ax = gs[ccd-1]

        in_ccd = ((common_wavelength > (3 + ccd) * 1000) & (common_wavelength < (4 + ccd) * 1000))

        ax.plot(common_wavelength[in_ccd],residual_flux_percentiles[1][in_ccd], 'k', label='Median Residual', lw = 0.75)
        ax.fill_between(common_wavelength[in_ccd], residual_flux_percentiles[0][in_ccd], residual_flux_percentiles[2][in_ccd], color = 'navajowhite', label=r'16th-84th Percentile', lw=0.75)

        ax.set_xlim(common_wavelength[in_ccd][0]-5, common_wavelength[in_ccd][-1]+5)
        ax.set_xlabel(r'Wavelength $\lambda_\mathrm{air}~/~\mathrm{\AA}$', fontsize=12)
        if ccd == 1:
            ax.set_ylabel(r'Residuals $\Delta f_\lambda$', fontsize=12)
            ax.legend(fontsize=12, loc='upper right', frameon=False)

    plt.tight_layout(w_pad=0, h_pad=0)
    plt.savefig('figures/residual_flux_percentiles_m1.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

    f, gs = plt.subplots(1,4,figsize=(15,3), sharey=True)
    for ccd in [1, 2, 3, 4]:
        
        ax = gs[ccd-1]

        in_ccd = ((common_wavelength > (3 + ccd) * 1000) & (common_wavelength < (4 + ccd) * 1000))

        ax.plot(common_wavelength[in_ccd],absolute_residual_flux_percentiles[1][in_ccd], 'k', label='Median Residual', lw = 0.75)
        ax.fill_between(common_wavelength[in_ccd], absolute_residual_flux_percentiles[0][in_ccd], absolute_residual_flux_percentiles[2][in_ccd], color = 'navajowhite', label=r'16th-84th Percentile', lw=0.75)

        ax.set_xlim(common_wavelength[in_ccd][0]-5, common_wavelength[in_ccd][-1]+5)
        ax.set_xlabel(r'Wavelength $\lambda_\mathrm{air}~/~\mathrm{\AA}$', fontsize=12)
        if ccd == 1:
            ax.set_ylabel(r'Residuals $\vert \Delta f_\lambda \vert$', fontsize=12)
            ax.legend(fontsize=12, loc='upper right', frameon=False)

    plt.tight_layout(w_pad=0, h_pad=0)
    plt.savefig('figures/absolute_residual_flux_percentiles_m1.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

    # Previous plot; replaced with version that includes significant and identified peaks below
    # f, gs = plt.subplots(1,4,figsize=(15,3), sharey=True)
    # for ccd in [1, 2, 3, 4]:
        
    #     ax = gs[ccd-1]

    #     in_ccd = ((common_wavelength > (3 + ccd) * 1000) & (common_wavelength < (4 + ccd) * 1000))

    #     ax.plot(common_wavelength[in_ccd],significance_residual_flux_percentiles[1][in_ccd], 'k', label='Median Residual', lw = 0.75)
    #     ax.fill_between(common_wavelength[in_ccd], significance_residual_flux_percentiles[0][in_ccd], significance_residual_flux_percentiles[2][in_ccd], color = 'navajowhite', label=r'16th-84th Percentile', lw=0.75)

    #     ax.set_xlim(common_wavelength[in_ccd][0]-5, common_wavelength[in_ccd][-1]+5)
    #     ax.set_xlabel(r'Wavelength $\lambda_\mathrm{air}~/~\mathrm{\AA}$', fontsize=12)
    #     if ccd == 1:
    #         ax.set_ylabel(r'Significance of $\vert \Delta f_\lambda \vert$', fontsize=12)
    #         ax.legend(fontsize=12, loc='upper right', frameon=False)

    #     ax.set_yscale('log')

    # plt.tight_layout(w_pad=0, h_pad=0)
    # plt.savefig('figures/significance_residual_flux_percentiles_m1.png', dpi=300, bbox_inches='tight')
    # plt.show()
    # plt.close()

In [ ]:
# Calculate how often residuals are above/below +/- a 1/2/3 sigma for model 1?

significance_percentages = dict()

for significance in [1,2,3]:

    # How often is residual above +X sigma?
    residuals_above_significance = model1_data['flux'] * np.sqrt(model1_data['ivar']) > significance
    percentage_above_significance = 100 * np.mean(residuals_above_significance, axis=0)
    significance_percentages[str(significance)+'_above'] = percentage_above_significance

    # How often is residual below -X sigma?
    residuals_below_significance = model1_data['flux'] * np.sqrt(model1_data['ivar']) < -significance
    percentage_below_significance = 100 * np.mean(residuals_below_significance, axis=0)
    significance_percentages[str(significance)+'_below'] = percentage_below_significance

In [ ]:
# More condensed visualisation of the significance of residuals for model 1 with identified peaks above tuned thresholds

rerun_everything = True

if rerun_everything:

    tuned_parameters = dict()
    tuned_parameters['threshold'] = 75
    tuned_parameters['peak_width'] = 3
    tuned_parameters['peak_distance'] = 6

    np.savetxt('tex_text/tuned_threshold.tex', [str(tuned_parameters['threshold'])+'%'], fmt='%s')
    np.savetxt('tex_text/tuned_peak_width.tex', [str(tuned_parameters['peak_width'])+'%'], fmt='%s')
    np.savetxt('tex_text/tuned_peak_distance.tex', [str(tuned_parameters['peak_distance'])+'%'], fmt='%s')

    minimum = dict()

    input_for_peaks = significance_residual_flux_percentiles[1,:].copy()
    for ccd in [1, 2, 3, 4]:
        in_ccd = ((common_wavelength > (3 + ccd) * 1000) & (common_wavelength < (4 + ccd) * 1000))
        minimum[str(ccd)] = np.percentile(input_for_peaks[in_ccd], tuned_parameters['threshold'])
        input_for_peaks[in_ccd & (input_for_peaks < minimum[str(ccd)])] = 0.0
        
    significance_peaks = find_peaks(input_for_peaks, width = tuned_parameters['peak_width'], distance = tuned_parameters['peak_distance'])
    print(f"Number of peaks with median residual flux significance above {tuned_parameters['threshold']}th percentile: {len(significance_peaks[0])}")
    np.savetxt('tex_text/number_significant_peaks.tex', [str(len(significance_peaks[0]))+'%'], fmt='%s')

    # create table with wavelength and significance of peaks
    significant_peaks_table = Table()
    significant_peaks_table['wave'] = common_wavelength[significance_peaks[0]]
    significant_peaks_table['significance'] = significance_residual_flux_percentiles[1,:][significance_peaks[0]]
    for fit_output in ['prominences', 'widths', 'width_heights']:
        significant_peaks_table[fit_output] = significance_peaks[1][fit_output]
    significant_peaks_table['left_bases'] = common_wavelength[significance_peaks[1]['left_bases']]
    significant_peaks_table['right_bases'] = common_wavelength[significance_peaks[1]['right_bases']]
    significant_peaks_table.write('data/significant_peaks_baseline_sample.fits', overwrite=True)

    f, gs = plt.subplots(1, 4,figsize=(15,3), sharey=True)

    for ccd in [1, 2, 3, 4]:   
        ax = gs[ccd-1]

        in_ccd = ((common_wavelength > (3 + ccd) * 1000) & (common_wavelength < (4 + ccd) * 1000))

        handle1 = ax.fill_between(
            common_wavelength[in_ccd],
            np.zeros_like(input_for_peaks[in_ccd]),
            significance_residual_flux_percentiles[1,:][in_ccd],         color = 'navajowhite',
            label = 'Median Residual Significance', lw = 0.5)
        handle3 = ax.fill_between(
            common_wavelength[in_ccd],
            np.zeros_like(input_for_peaks[in_ccd]),
            input_for_peaks[in_ccd],
            lw = 0.5,
            color = 'C0', 
            label = 'Above Threshold'
            )
        ax.plot(common_wavelength[in_ccd],significance_residual_flux_percentiles[1,:][in_ccd], lw = 0.5, c = 'k')
        handle2 = ax.axhline(minimum[str(ccd)], color='k', lw=0.5, ls='--', label = 'Thresholds in CCDs: '+"/".join([f'{minimum[str(ccd)]:.2f}' for ccd in [1,2,3,4]]))
        first_line = True; peaks_in_ccd = 0
        for peak in significance_peaks[0]:
            if (peak >= np.where(in_ccd)[0][0]) & (peak <= np.where(in_ccd)[0][-1]):
                if first_line:
                    label = r'Significant Found Peaks'
                    first_line = False
                    handle4 = ax.axvline(common_wavelength[peak], color='red', lw=0.5, ls='--', label = label, ymin=0.6, ymax=0.8)
                else:
                    label = '_nolegend_'
                    ax.axvline(common_wavelength[peak], color='red', lw=0.5, ls='--', label = label, ymin=0.6, ymax=0.8)
                peaks_in_ccd += 1

        print(f"Number of peaks with median residual flux significance above {tuned_parameters['threshold']}th percentile in CCD {ccd}: {peaks_in_ccd}")
        np.savetxt(f'tex_text/number_significant_peaks_ccd{ccd}.tex', [str(peaks_in_ccd)+'%'], fmt='%s')

        ax.set_xlim(common_wavelength[in_ccd][0]-5, common_wavelength[in_ccd][-1]+5)
        if ccd == 1:
            ax.set_ylabel(r'Significance of $\vert \Delta f_\lambda \vert$', fontsize=12)
            ax.legend(handles = [handle1], fontsize=10, loc='upper center', handlelength=1)
        if ccd == 2:
            ax.legend(handles = [handle2], fontsize=10, loc='upper center', handlelength=1)
        if ccd == 3:
            ax.legend(handles = [handle3], fontsize=10, loc='upper right', handlelength=1)
        if ccd == 4:
            ax.legend(handles = [handle4], fontsize=10, loc='upper center', handlelength=1)
        ax.set_xlabel(r'Wavelength $\lambda_\mathrm{air}~/~\mathrm{\AA}$', fontsize=12)

    ax.set_yscale('log')
    plt.tight_layout(w_pad=-2, h_pad=0)
    plt.savefig('figures/significance_residual_flux_percentiles_m1.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
# Plot a larger version to inspect
if rerun_everything:

    f, gs = plt.subplots(4,1,figsize=(15,10), sharey=True)

    for ccd in [1, 2, 3, 4]:   
        ax = gs[ccd-1]

        in_ccd = ((common_wavelength > (3 + ccd) * 1000) & (common_wavelength < (4 + ccd) * 1000))

        ax.fill_between(
            common_wavelength[in_ccd],
            np.zeros_like(input_for_peaks[in_ccd]),
            significance_residual_flux_percentiles[1,:][in_ccd],         color = 'navajowhite',
            label = 'Median Significance of Residuals', lw = 0.5)
        ax.fill_between(
            common_wavelength[in_ccd],
            np.zeros_like(input_for_peaks[in_ccd]),
            input_for_peaks[in_ccd],
            lw = 0.5)
        ax.plot(common_wavelength[in_ccd],significance_residual_flux_percentiles[1,:][in_ccd], lw = 0.5, c = 'k')
        ax.axhline(minimum[str(ccd)], color='k', lw=0.5, ls='--', label = f'{tuned_parameters["threshold"]}th percentile of significances in CCDs 1/2/3/4: '+"/".join([f'{minimum[str(ccd)]:.2f}' for ccd in [1,2,3,4]]))
        first_line = True
        for peak in significance_peaks[0]:
            if (peak >= np.where(in_ccd)[0][0]) & (peak <= np.where(in_ccd)[0][-1]):
                if first_line:
                    label = 'Significant Found Peaks'
                    first_line = False
                else:
                    label = '_nolegend_'
                ax.axvline(common_wavelength[peak], color='red', lw=0.5, ls='--', label = label)
        ax.set_xlim(common_wavelength[in_ccd][0]-5, common_wavelength[in_ccd][-1]+5)
        if ccd == 4:
            ax.set_xlabel(r'Wavelength $\lambda_\mathrm{air}~/~\mathrm{\AA}$', fontsize=12)
        if ccd == 1:
            ax.legend(fontsize=12, loc='upper center', ncol=4)
        ax.set_ylabel(r'Significance of $\vert \Delta f_\lambda \vert$', fontsize=12)
            
    ax.set_yscale('log')
    plt.tight_layout(w_pad=0, h_pad=0)
    plt.savefig('figures/significant_residual_peaks_m1_enlarged.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
# Where is the observation more than 50% more shallower
peaks_more_than_50percent_shallower = find_peaks(percentage_above_significance, height=50)[0]
print(f"Wavelengths of {len(peaks_more_than_50percent_shallower)} pixels with residual flux significance above +1 sigma in more than 50% of spectra:")
for index in peaks_more_than_50percent_shallower:
    print(f"{common_wavelength[index]:.2f} Å")

# Where is the observation more than 50% more deeper
peaks_more_than_50percent_deeper = find_peaks(percentage_below_significance, height=50)[0]
print(f"Wavelengths of {len(peaks_more_than_50percent_deeper)} pixels with residual flux significance below -1 sigma in more than 50% of spectra:")
for index in peaks_more_than_50percent_deeper:
    print(f"{common_wavelength[index]:.2f} Å")

In [ ]:
# Plot how often residuals are above/below +/- a 1/2/3 sigma for model 1
rerun_everything = True
if rerun_everything:
    f, gs = plt.subplots(1,4,figsize=(15,3), sharey=True)

    for ccd in [1, 2, 3, 4]:
        
        ax = gs[ccd-1]

        in_ccd = ((common_wavelength > (3 + ccd) * 1000) & (common_wavelength < (4 + ccd) * 1000))

        ax.fill_between(
            common_wavelength[in_ccd],
            np.zeros(len(common_wavelength[in_ccd])),
            percentage_above_significance[in_ccd],
            color = 'lightblue'#, 'C0', label='Too high', lw = 0.75
        )
        ax.fill_between(
            common_wavelength[in_ccd],
            np.zeros(len(common_wavelength[in_ccd])),
            -percentage_below_significance[in_ccd],
            color = 'lightcoral'#, 'C3', label='Too low', lw = 0.75
        )
        line1, = ax.plot(common_wavelength[in_ccd],percentage_above_significance[in_ccd], 'C0', label='Observation $>' + str(significance) + r'\sigma$ shallower', lw = 0.75)
        line2, = ax.plot(common_wavelength[in_ccd],-percentage_below_significance[in_ccd], 'C3', label='Observation $>' + str(significance) + r'\sigma$ deeper', lw = 0.75)

        ax.set_xlim(common_wavelength[in_ccd][0]-5, common_wavelength[in_ccd][-1]+5)
        ax.set_xlabel(r'Wavelength $\lambda_\mathrm{air}~/~\mathrm{\AA}$', fontsize=12)
        if ccd == 1:
            ax.set_ylabel('Percentage of spectra\n above ' + r'$'+ str(significance) + r'\sigma$ or below ' + r'$-'+ str(significance) + r'\sigma$', fontsize=12)
        if ccd == 1:
            legend1 = ax.legend(handles=[line1], fontsize=12, loc='upper center', borderpad=0.25, edgecolor='none', framealpha=0.5)
            ax.add_artist(legend1)
            legend2 = ax.legend(handles=[line2], fontsize=12, loc='lower center', borderpad=0.25, edgecolor='none', framealpha=0.5)
            ax.add_artist(legend2)

    plt.tight_layout(w_pad=0, h_pad=0)
    plt.savefig(f'figures/percentage_{significance}sigma_residual_flux_percentiles_m1.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
# Plot cumulative distribution of significance of residual flux; with an inset of the typical residuals.

if rerun_everything:
    f, ax = plt.subplots(figsize=(6,4))

    ax.ecdf(significance_residual_flux_percentiles[1].clip(max=4), label='Median Residual', c = 'k')
    ax.ecdf(significance_residual_flux_percentiles[0].clip(max=4), label='16th Percentile', c = 'navajowhite')
    ax.ecdf(significance_residual_flux_percentiles[2].clip(max=4), label='84th Percentile', c = 'lightcoral')
    ax.text(4.0,0.95,r'clipped at 4$\sigma$',va='top',ha='right',fontsize=10)
    ax.set_xlabel(r'Significance of $\vert \Delta f_\lambda \vert$', fontsize=12)
    ax.set_ylabel('Cumulative Distribution', fontsize=12)
    ax.legend(fontsize=12, loc=(0.6,0.554), frameon=False)

    ax.set_xlim(0, 4.05)
    ax.set_ylim(0, 1.0)

    x = 0.878
    ax.axvline(1,color='grey', ls='--', lw=0.75, alpha=0.5)
    ax.axhline(x,color='grey', ls='--', lw=0.75, alpha=0.5)
    ax.text(1.01, x-0.01, '(1,0.878)', ha='left', va='top', fontsize=10)

    x = 0.9734
    ax.axvline(2,color='grey', ls='--', lw=0.75, alpha=0.5)
    ax.axhline(x,color='grey', ls='--', lw=0.75, alpha=0.5)
    ax.text(2.01, x-0.01, '(2,0.973)', ha='left', va='top', fontsize=10)

    ax.axvline(3,color='grey', ls='--', lw=0.75, alpha=0.5)
    ax.axvline(4,color='grey', ls='--', lw=0.75, alpha=0.5)

    # inset with cumulative distribution oftypical absolute residuals
    axins = ax.inset_axes([0.617, 0.15, 0.37, 0.37])
    axins.ecdf(absolute_residual_flux_percentiles[1].clip(max=0.1), label='Median Residual', c = 'k')
    axins.ecdf(absolute_residual_flux_percentiles[0].clip(max=0.1), label='16th Percentile', c = 'navajowhite')
    axins.ecdf(absolute_residual_flux_percentiles[2].clip(max=0.1), label='84th Percentile', c = 'lightcoral')
    axins.text(0.10,0.95,'clipped\nat 0.1',va='top',ha='right',fontsize=8)
    axins.set_xlabel(r'Absolute Residuals $\vert \Delta f_\lambda \vert$', fontsize=10)
    axins.set_ylabel(r'Cumulative Distribution', fontsize=10)
    axins.legend(fontsize=7, loc='lower right', frameon=False)
    axins.tick_params(axis='both', labelsize=9)    
    axins.set_xlim(0, 0.105)
    axins.set_yticks([0,0.5,1])
    axins.set_ylim(0, 1.0)
    axins.axhline(0.5, color='grey', ls='--', lw=0.75, alpha=0.5)

    plt.tight_layout()
    plt.savefig('figures/cdf_significance_residual_flux_percentiles_m1.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

### 3.1.2 Identifying possible causes of residuals

In [ ]:
# Crossmatch significant peaks with line lists by Hinkle+ 2000

if rerun_everything:
    # Hinkle+2000 data atomic line data from their Appendix I
    with open('data/hinkle_2000vnia.book.....H_atlas_appi', 'r') as f:
        lines = f.readlines()

    hinkle_lines_species = []
    hinkle_lines_wavelength = []

    for i in range(1,len(lines)):
        line = lines[i]
        line = line.replace('\n','')
        line = line.replace(' ','')
        if len(line) > 0:
            try:
                line = float(line.split(',')[0])
                hinkle_lines_species.append(species)
                hinkle_lines_wavelength.append(line)
            except:
                species = line

    hinkle_lines_table = Table()
    hinkle_lines_table['species'] = hinkle_lines_species
    hinkle_lines_table['wave_air'] = hinkle_lines_wavelength
    hinkle_lines_table.write('data/hinkle_2000vnia.book.....H_atlas_appi.fits', overwrite=True)

In [ ]:
# Load significant peaks and Hinkle+ 2000 line list and spectra for plotting
significant_peaks_table = Table.read('data/significant_peaks_baseline_sample.fits')
hinkle_lines_table = Table.read('data/hinkle_2000_atlas/hinkle_2000_APPI_lines.txt', format = 'ascii', names=['species', 'wave_air'])
hinkle_spectra = Table.read('data/hinkle_2000_atlas/hinkle_2000_solar_arcturus_telluric_atlas.fits')

# Read in 307002 lines from GALAH DR4 lineslist
galah_dr4_linelist = Table.read('data/galah_master_v5.2.fits')
# Let's only look at 2622 lines with log_gf_ref entries & DEPTH > 0.85
galah_dr4_linelist = galah_dr4_linelist[(
    # (~galah_dr4_linelist['LOG_GF_REF'].mask) &
    (galah_dr4_linelist['NAME'][:,1] != 'H ') & (galah_dr4_linelist['NAME'][:,1] != 'C ') & (galah_dr4_linelist['NAME'][:,1] != 'N ') & (galah_dr4_linelist['NAME'][:,1] != 'O ')
    # (galah_dr4_linelist['DEPTH'] > 0.85)
)]
galah_dr4_linelist['NAME'] = np.array(galah_dr4_linelist['NAME'], dtype='str')

print('Loaded Peaks Table, Hinkle+2000 atlases and lines lines, and GALAH DR4 linelist ('+str(len(galah_dr4_linelist))+' lines')# with log_gf_ref entries and depth > 0.85)')

In [ ]:
# Create figures for inspecting Hinkle+2000 atlases of Sun and Arcuturus + significant line.

def set_line_window_ticks(ax, lambda0):
    candidate_offsets = [
        np.array([-0.4, -0.2, 0.0, 0.2, 0.4]),
        np.array([-0.3, -0.1, 0.1, 0.3, 0.5]),
        np.array([-0.2, 0.0, 0.2, 0.4, 0.6]),
    ]

    xmin, xmax = ax.get_xlim()

    for offsets in candidate_offsets:
        ticks = lambda0 + offsets
        if np.all((ticks > xmin) & (ticks < xmax)):
            ax.set_xticks(ticks)
            return ticks

for peak_index, significant_peak in enumerate(significant_peaks_table):

    if "{:.2f}".format(significant_peak['wave']) in ['4727.26','7799.96','4731.45','4861.49','5780.85','7853.25']:

        window = 1.1

        hinkle_region = (hinkle_spectra['WAVELENGTH'] > significant_peak['wave']-window/2.0) & (hinkle_spectra['WAVELENGTH'] < significant_peak['wave']+window/2.0)
        hinkle_lines_in_region = (hinkle_lines_table['wave_air'] > significant_peak['wave']-window/2.1) & (hinkle_lines_table['wave_air'] < significant_peak['wave']+window/2.1)
        residual_region = (common_wavelength > significant_peak['wave']-window/2.0) & (common_wavelength < significant_peak['wave']+window/2.0)
        dr4_linelist_in_region = (galah_dr4_linelist['LAMBDA'] > significant_peak['wave']-window/2.1) & (galah_dr4_linelist['LAMBDA'] < significant_peak['wave']+window/2.1) & (galah_dr4_linelist['DEPTH'] > 0.85)

        f, ax = plt.subplots(figsize=(5,4))

        dr4_5_95 = ax.fill_between(
            common_wavelength[residual_region],
            flux_percentiles[0][residual_region],
            flux_percentiles[4][residual_region],
            color = 'lightgrey',
            lw = 0.5,
            alpha = 0.5,
            label = 'DR4 Obs. 5th-95th Perc.'
        )
        dr4_16_84 = ax.fill_between(
            common_wavelength[residual_region],
            flux_percentiles[1][residual_region],
            flux_percentiles[3][residual_region],
            color = 'grey',
            lw = 0.5,
            alpha = 0.5,
            label = 'DR4 Obs. 16th-84th Perc.'
        )
        dr4_50, = ax.plot(
            common_wavelength[residual_region],
            flux_percentiles[2][residual_region],
            c = 'k',
            lw = 0.5,
            label = 'DR4 Obs. Median'
        )

        l3, = ax.plot(
            hinkle_spectra['WAVELENGTH'][hinkle_region],
            hinkle_spectra['SOLARFLUX'][hinkle_region],
            c = 'k',
            ls = 'dashed',
            lw = 0.5,
            label = 'Sun'
        )
        l4, = ax.plot(
            hinkle_spectra['WAVELENGTH'][hinkle_region],
            hinkle_spectra['ARCTURUS'][hinkle_region],
            c = 'k',
            ls = 'dotted',
            lw = 0.5,
            label = 'Arcturus'
        )
        l5, = ax.plot(
            hinkle_spectra['WAVELENGTH'][hinkle_region],
            hinkle_spectra['TELLURIC'][hinkle_region],
            c = 'darkblue',
            lw = 0.5,
            label = 'Telluric'
        )

        l1 = ax.fill_between(
            common_wavelength[residual_region],
            -0.2*np.ones(len(common_wavelength[residual_region])),
            np.log10(significance_residual_flux_percentiles[1][residual_region]),
            color = 'navajowhite',
            lw = 0.5,
            alpha = 0.2,
            label = r'$\log_{10}(\mathrm{Significance})$',
            zorder = 4
        )
        ax.plot(
            common_wavelength[residual_region],
            np.log10(significance_residual_flux_percentiles[1][residual_region]),
            c = 'navajowhite',
            lw = 1,
            label = '_nolegend_',
            zorder = 3
        )

        l21 = ax.fill_between(
            common_wavelength[residual_region],
            0.5 + np.zeros_like(common_wavelength[residual_region]),
            0.5 + percentage_above_significance[residual_region] / 200.,
            color = 'lightblue',
            lw = 1,
            alpha = 0.25,
            zorder = 1,
            label = 'DR4 Obs. shallower'
        )
        l22 = ax.fill_between(
            common_wavelength[residual_region],
            0.5 + np.zeros_like(common_wavelength[residual_region]),
            0.5 - percentage_below_significance[residual_region] / 200.,
            color = 'lightcoral',
            lw = 1,
            alpha = 0.25,
            zorder = 2,
            label = 'DR4 Obs. deeper'
        )

        l4 = ax.fill_between(
            common_wavelength[residual_region],
            -0.2 + np.zeros_like(common_wavelength[residual_region]),
            -0.2 + np.log10(reduction_sky_tellurics['sky'][residual_region]),
            color = 'darkblue',
            lw = 0.5,
            alpha = 0.5,
            label = 'log10(Sky)'
        )

        l2 = ax.axvline(significant_peak['wave'], color='red', lw=1, ls='--', ymin = 0.0, ymax=0.7, label = 'Significant Peak\nat '+r'$\lambda_\mathrm{air}=$'+"{:.2f}".format(significant_peak['wave'])+r'$\mathrm{\AA}$')
        ax.text(significant_peak['wave'], -0.1, "{:.2f}".format(significant_peak['wave'])+r'$\mathrm{\AA}$', rotation=90, va='bottom', ha='left', fontsize=9, color='red')

        for hinkle_line in hinkle_lines_table[hinkle_lines_in_region]:
            ax.axvline(hinkle_line['wave_air'], color='grey', lw=1, alpha=0.6, ymin=0.7, ymax=0.75)
            ax.text(hinkle_line['wave_air'], 1.01, hinkle_line['species'], rotation=90, va='bottom', ha='center', fontsize=9, color='k')

        first_line = True
        for galah_line in galah_dr4_linelist[dr4_linelist_in_region]:
            name = str(galah_line['NAME'][0]).replace(' ','')
            ion = galah_line['ION']
            if ion == 1:
                name += ' I'
            elif ion == 2:
                name += ' II'

            if first_line:
                label = 'Strong DR4 Lines'
                l0 = ax.axvline(galah_line['LAMBDA'], color='C0', lw=1, alpha=0.65, ymin=0.75, ymax=0.8, label = label)
                first_line = False
            else:
                label = '_nolegend_'
                ax.axvline(galah_line['LAMBDA'], color='C0', lw=1, alpha=0.65, ymin=0.75, ymax=0.8, label = label)
            
            ax.text(galah_line['LAMBDA'], 1.15, name, rotation=90, va='bottom', ha='center', fontsize=9, color='C0')

        leg1 = ax.legend(handles = [l0, l1, l21, l22], ncol=1, fontsize=8, loc='lower left', framealpha=0.9)
        ax.add_artist(leg1)
        leg2 = ax.legend(handles = [l3,l4, l5], ncol=1, fontsize=8, loc='lower right', framealpha=0.9)
        leg2.set_title('Hinkle+2000', prop={'size': 8})
        
        ax.set_xlabel(r'Wavelength $\lambda_\mathrm{air}~/~\mathrm{\AA}$', fontsize=12)
        ax.set_ylabel('Fluxes / a.u.', fontsize=12)
        ax.set_xlim(significant_peak['wave']-window/2.0, significant_peak['wave']+window/2.0)
        ax.set_ylim(-0.2,1.3)

        letter = ''
        if "{:.2f}".format(significant_peak['wave']) == '4727.26':
            letter = 'A: '
        if "{:.2f}".format(significant_peak['wave']) == '7799.96':
            letter = 'B: '
        if "{:.2f}".format(significant_peak['wave']) == '4731.45':
            letter = 'C: '
        if "{:.2f}".format(significant_peak['wave']) == '4861.49':
            letter = 'D: '
        if "{:.2f}".format(significant_peak['wave']) == '5780.85':
            letter = 'E: '
        if "{:.2f}".format(significant_peak['wave']) == '7853.25':
            letter = 'F: '

        ax.set_title(letter+'GALAH DR4 Residual at '+r'$\lambda_\mathrm{air}=$'+"{:.2f}".format(significant_peak['wave'])+r'$\mathrm{\AA}$', fontsize=12, pad=10)
        
        plt.tight_layout()
        plt.savefig('residual_inspection/residual_region_'+"{:.2f}".format(significant_peak['wave'])+'.pdf', dpi=300, bbox_inches='tight')
        # if peak_index < 2:  # Show the first few
        plt.show()
    plt.close()

In [ ]:
# Match residuals with sky lines from DR4 observation

reduction_sky_tellurics = Table.read('data/reduction_sky_tellurics.fits')

sky_line_peaks, sky_line_peaks_strength = find_peaks(reduction_sky_tellurics['sky'], height = 2)
sky_line_wave = reduction_sky_tellurics['wave'][sky_line_peaks]
sky_lines = Table()
sky_lines['wave'] = np.round(sky_line_wave, 3)
sky_lines['strength'] = np.round(sky_line_peaks_strength['peak_heights'],1)

In [ ]:
# Interstellar features - based on Vogroncic et al. (2023,https://ui.adsabs.harvard.edu/abs/2023MNRAS.521.3727V)
# DIB catalogue with EW[@E(B-V)=0] >= 0.01AA or EW/E(B-V) > 0.025AA & including KI interstellar line at 7698.9643AA

vogroncic_dib_lines = np.array([
    4726.40, 4728.28,
    4762.57, # <- strong
    4848.77,4855.25,
    4859.89,4866.65,
    5705.21, # <- strong
    5735.22,5741.28,
    5747.62, # <- strong
    5753.17,
    5776.03,
    5780.59, # <- strong
    5784.78, # <- strong
    5797.19, # <- strong
    5809.42,
    5843.54,
    5849.91,
    6493.11, # <- strong
    6496.67, # <- strong
    6516.33,
    6526.11,
    6530.17, # <- strong
    6535.89,
    6570.83,
    6589.97, # <- strong
    6592.53,
    6613.66, # <- strong
    6699.33,
    7698.9643
    ])

In [ ]:
f, ax = plt.subplots(figsize=(15,3))
l1 = ax.fill_between(
    common_wavelength,
    percentage_above_significance,
    np.zeros_like(common_wavelength),
    color = 'lightblue',
    lw = 0.5, label = r'Observation $> 3\sigma$ shallower'
)
l2 = ax.fill_between(
    common_wavelength,
    np.zeros_like(common_wavelength),
    -percentage_below_significance,
    color = 'lightcoral',
    lw = 0.5, label = r'Observation $> 3\sigma$ deeper'
)

ax2 = ax.twinx()
l3, = ax2.plot(
    common_wavelength,
    reduction_sky_tellurics['sky'],
    #np.log10(reduction_sky_tellurics['sky']),
    c = 'darkblue',
    lw = 1,
    label = r'$f_\mathrm{sky}$'
)

ax2.legend(handles = [l1, l2, l3], ncol=3, fontsize=10, loc='upper center')
ax.set_xlim(7690,7875)
ax.set_ylim(-25,25)
ax.set_xlabel(r'Wavelength $\lambda_\mathrm{air}~/~\mathrm{\AA}$', fontsize=12)
ax.set_ylabel('Percentage of spectra\n'+r'above $3\sigma$ or below $-3\sigma$', fontsize=12)
ax2.set_ylabel(r'$f_\mathrm{sky}$ / a.u.', fontsize=12)
plt.tight_layout()
plt.savefig('figures/skyline_confirmation.pdf', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
peak_inspection = dict()

for peak_index, significant_peak in enumerate(significant_peaks_table):

    # Column 1: Peak index
    peak_inspection.setdefault('peak_index', []).append(peak_index)

    # Column 2: Wavelength of the residual
    peak_inspection.setdefault('peak_wave_air', []).append("{:.2f}".format(significant_peak['wave']))

    # Column 3: Significance of the residual at that wavelength
    peak_inspection.setdefault('significance', []).append(np.round(significant_peak['significance'],1))

    # Column 4: Peak Classification (placeholder for later)
    peak_inspection.setdefault('classification', []).append('Not Yet Classified')

    # Column 5: Percentage of spectra with observation being more than 3 sigma shallower at that wavelength
    percentage_shallower = percentage_above_significance[np.where(common_wavelength == significant_peak['wave'])[0][0]]
    peak_inspection.setdefault('percent_shallower', []).append(int(np.round(percentage_shallower)))

    # Column 7: Percentage of spectra with observation being more than 3 sigma deeper at that wavelength
    percentage_deeper = percentage_below_significance[np.where(common_wavelength == significant_peak['wave'])[0][0]]
    peak_inspection.setdefault('percent_deeper', []).append(int(np.round(percentage_deeper)))

    # Columns 8-11: Is there a Hinkle+2000 line within 0.05 Å of the peak?
    nearby_hinkle_lines = hinkle_lines_table[np.abs(hinkle_lines_table['wave_air'] - significant_peak['wave']) < 0.05]
    if len(nearby_hinkle_lines) > 0:
        closest_hinkle_line = nearby_hinkle_lines[np.argmin(np.abs(nearby_hinkle_lines['wave_air'] - significant_peak['wave']))]
        peak_inspection.setdefault('hinkle_species', []).append(closest_hinkle_line['species'])
        peak_inspection.setdefault('hinkle_wave_air', []).append(closest_hinkle_line['wave_air'])
        peak_inspection.setdefault('hinkle_distance', []).append(np.round(np.abs(closest_hinkle_line['wave_air'] - significant_peak['wave']),2))
    else:
        peak_inspection.setdefault('hinkle_species', []).append('None')
        peak_inspection.setdefault('hinkle_wave_air', []).append(np.nan)
        peak_inspection.setdefault('hinkle_distance', []).append(np.nan)

    # Columns 12-16: Is there a GALAH DR4 line within 0.05 Å of the peak, and if so: what is the loggf?
    nearby_dr4_lines = galah_dr4_linelist[np.abs(galah_dr4_linelist['LAMBDA'] - significant_peak['wave']) < 0.05]
    if len(nearby_dr4_lines) > 0:
        closest_dr4_line = nearby_dr4_lines[np.argmin(np.abs(nearby_dr4_lines['LAMBDA'] - significant_peak['wave']))]
        peak_inspection.setdefault('dr4_line_name', []).append(closest_dr4_line['NAME'][0]+(' I' if closest_dr4_line['ION'] == 1 else ' II' if closest_dr4_line['ION'] == 2 else ''))
        peak_inspection.setdefault('dr4_wave_air', []).append(closest_dr4_line['LAMBDA'])
        peak_inspection.setdefault('dr4_distance', []).append(np.round(np.abs(closest_dr4_line['LAMBDA'] - significant_peak['wave']),2))
        peak_inspection.setdefault('dr4_log_gf_ref', []).append(closest_dr4_line['LOG_GF_REF'].replace(' ',''))
        peak_inspection.setdefault('dr4_log_gf', []).append(closest_dr4_line['LOG_GF'])
        peak_inspection.setdefault('dr4_depth', []).append(closest_dr4_line['DEPTH'])
    else:
        peak_inspection.setdefault('dr4_line_name', []).append('None')
        peak_inspection.setdefault('dr4_wave_air', []).append(np.nan)
        peak_inspection.setdefault('dr4_distance', []).append(np.nan)
        peak_inspection.setdefault('dr4_log_gf_ref', []).append('--')
        peak_inspection.setdefault('dr4_log_gf', []).append(np.nan)
        peak_inspection.setdefault('dr4_depth', []).append(np.nan)

    # Column 18: Width
    peak_inspection.setdefault('width', []).append(np.round(significant_peak['widths'],1))

    # Column 19-20: Skyline within 1Å?
    nearby_skyline = sky_lines[np.abs(sky_lines['wave'] - significant_peak['wave']) < 1.0]
    if len(nearby_skyline) > 0:
        closest_skyline = nearby_skyline[np.argmin(np.abs(nearby_skyline['wave'] - significant_peak['wave']))]
        peak_inspection.setdefault('skyline_wave', []).append(closest_skyline['wave'])
        peak_inspection.setdefault('skyline_strength', []).append(closest_skyline['strength'])
        peak_inspection.setdefault('skyline_distance', []).append(np.round(np.abs(closest_skyline['wave'] - significant_peak['wave']),2))
    else:
        peak_inspection.setdefault('skyline_wave', []).append(np.nan)
        peak_inspection.setdefault('skyline_strength', []).append(np.nan)
        peak_inspection.setdefault('skyline_distance', []).append(np.nan)

peak_inspection = Table(peak_inspection)
# Column 21: Classification comments?
peak_inspection['comments'] = np.full(len(peak_inspection), '--', dtype='U200')

"""
THIS IS WHERE WE CLASSIFY AUTOMATICALLY
"""
automatically_classified_peaks = 0

for peak_index, row in enumerate(peak_inspection):

    # Classifications 1&2: "too low loggf" and "too high loggf" - based on consistently identified lines in Hinkle+2000 and GALAH DR4
    if row['hinkle_species'] == row['dr4_line_name'] != 'None':

        # print(row['peak_index'], row['peak_wave_air'], row['percent_deeper'], row['percent_shallower'], row['hinkle_wave_air'], row['hinkle_species'], row['dr4_line_name'], row['dr4_wave_air'], np.round(row['dr4_depth'], 2), np.round(row['dr4_log_gf'], 2), row['dr4_log_gf_ref'])

        # Assess if the residuals are overwhelmingly shallower (loggf too high) or deeper (logff too low) than the model
        if row['percent_shallower'] > row['percent_deeper']:
            peak_inspection['classification'][peak_index] = 'loggf too high'
            automatically_classified_peaks += 1
            # print('#',row['peak_wave_air'],', loggf too high for ', row['hinkle_species'], row['percent_shallower'], 'vs.', row['percent_deeper'], '->', row['percent_shallower'] - row['percent_deeper'])
        elif row['percent_deeper'] >= row['percent_shallower']:
            peak_inspection['classification'][peak_index] = 'loggf too low'
            automatically_classified_peaks += 1
            # print('#',row['peak_wave_air'],', loggf too low for ', row['hinkle_species'], row['percent_deeper'], 'vs.', row['percent_shallower'], '->', row['percent_deeper'] - row['percent_shallower'])
        else:
            peak_inspection['comments'][peak_index] += 'Both deeper and shallower'
            # print(row['peak_index'], row['peak_wave_air'], row['percent_deeper'], row['percent_shallower'], row['hinkle_wave_air'], row['hinkle_species'], row['dr4_line_name'], row['dr4_wave_air'], np.round(row['dr4_depth'], 2), np.round(row['dr4_log_gf'], 2), row['dr4_log_gf_ref'])

    # Classification 3: "known missing" - lines in Hinkle+2000 but not in GALAH DR4
    elif row['dr4_line_name'] == 'None' and row['hinkle_species'] != 'None':
        peak_inspection['classification'][peak_index] = 'known missing'
        automatically_classified_peaks += 1
        peak_inspection['comments'][peak_index] += row['hinkle_species']+' in Hinkle+2000, but not DR4.'
        # print('#',row['peak_wave_air'],', loggf too low for ', row['hinkle_species'], row['percent_deeper'], 'vs.', row['percent_shallower'], '->', row['percent_deeper'] - row['percent_shallower'])
    

    # Classification 4: Interstellar features - based on Vogroncic+2023 DIB catalogue with EW[@E(B-V)=0] >= 0.01AA or EW/E(B-V) > 0.025AA with Q+ & including KI interstellar line at 7698.9643AA
    # if  :
    check_nearby_ism = np.where(np.abs(vogroncic_dib_lines - float(row['peak_wave_air'])) < 0.5)[0]
    if len(check_nearby_ism) > 0:
        if (row['width'] > 6):
            peak_inspection['classification'][peak_index] = 'interstellar'
            automatically_classified_peaks += 1
            # print('#',row['peak_wave_air'],', with width ', row['width'], 'Deeper ', row['percent_deeper'], 'Shallower', row['percent_shallower'], ' ISM feature at ', vogroncic_dib_lines[check_nearby_ism[0]], ' (Vogroncic+2023 DIB catalogue or KI interstellar line)')
        # else:
        #     print('# NO',row['peak_wave_air'],', with width ', row['width'], 'Deeper ', row['percent_deeper'], 'Shallower', row['percent_shallower'], ' ISM feature at ', vogroncic_dib_lines[check_nearby_ism[0]], ' (Vogroncic+2023 DIB catalogue or KI interstellar line)')
        # peak_inspection['comments'][peak_index] += 'ISM feature ('+str(vogroncic_dib_lines[check_nearby_ism[0]])+' with width ' + "{:.0f}".format(row['width']) + 'px) within 0.05 AA.'

    # Classification 5: Overwrite skylines
    if abs(row['skyline_distance']) < 0.5:
        if row['peak_wave_air'] not in ['6498.95','6522.10','6533.40','6553.28','6568.99','6577.63','6596.50','7715.22','7723.23','7751.09','7780.56','7840.83']:
            peak_inspection['classification'][peak_index] = 'Skyline'
            automatically_classified_peaks += 1
            # print('#',row['peak_wave_air'],', Skyline at ', row['skyline_wave'], ' (strength ', row['skyline_strength'], ')')
        else:
            peak_inspection['comments'][peak_index] += 'Weak Skyline within 0.5 AA'
    if row['peak_wave_air'] in ['7711.76','7712.72']:
        peak_inspection['classification'][peak_index] = 'Skyline'

print(automatically_classified_peaks, "out of", len(peak_inspection), "peaks automatically classified based on consistent Hinkle+2000 and GALAH DR4 line identifications and residual flux behavior.")

"""
THIS IS WHERE WE ACTUALLY DO THE WORK TO CORRECT THE AUTOMATIC ENTRIES!

for row in peak_inspection:
    print("peak_inspection["+str(row['peak_wave_air'])+"]['peak_classification'] = 'Unclassified'; peak_inspection["+str(row['peak_wave_air'])+"]['comments'] = '--'")
"""

peak_inspection['classification'][peak_inspection['peak_wave_air'] == '4861.49'] = 'Line-Formation'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '4861.49'] += 'Balmer Line, likely 3D non-LTE and chromospheric effects.'
peak_inspection['classification'][peak_inspection['peak_wave_air'] == '6563.12'] = 'Line-Formation'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '6563.12'] += 'Balmer Line, likely 3D non-LTE and chromospheric effects.'

peak_inspection['classification'][peak_inspection['peak_wave_air'] == '4730.99'] = 'Unclear'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '4730.99'] += 'Closest FeI line (more than 0.05AA away) is a blend of this line.'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '4773.95'] += 'Both CeII and blended FeI have too low loggf value.'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '4792.31'] += 'Two blended SiII lines have too high loggf value.'
peak_inspection['classification'][peak_inspection['peak_wave_air'] == '4796.26'] = 'Unclear'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '4796.26'] += 'Blend with unidentified line, closest CoI (more than 0.05AA away) has too low loggf.'
peak_inspection['classification'][peak_inspection['peak_wave_air'] == '4799.85'] = 'Unclear'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '4799.85'] += 'Blend with unidentified line, closest TiI (more than 0.05AA away) has too low loggf.'
peak_inspection['classification'][peak_inspection['peak_wave_air'] == '4805.00'] = 'Unclear'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '4805.00'] += 'Blend with unidentified line, closest TiII (more than 0.05AA away)'
peak_inspection['classification'][peak_inspection['peak_wave_air'] == '4812.36'] = 'Unclear'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '4812.36'] += 'Unidentified line is blending closest CrII line'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '4829.02'] += 'Either NiI or CeII have too high loggf value.'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '4836.84'] += 'Either NiI or CrI have too high loggf value.'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '4846.40'] += 'Either CrI or FeI have too high loggf value.'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '4851.46'] += 'All of Vi, CrI, and TiI have low high loggf values.'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '4884.91'] += 'CrI and likely TiI have low high loggf values.'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '4889.00'] += 'FeI has too high loggf value; possible blended (assymetric residual).'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '5754.21'] += 'Either SiI or VI have too high loggf value.'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '5762.91'] += 'FeI to the left has too low loggf value.'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '5780.85'] += 'Both FeI and TiI have too low loggf value.'
peak_inspection['classification'][peak_inspection['peak_wave_air'] == '5785.55'] = 'Unclear'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '5785.55'] += 'Closest MgI line (more than 0.05AA away) is a blend of this line.'
peak_inspection['classification'][peak_inspection['peak_wave_air'] == '5816.29'] = 'Unclear'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '5816.29'] += 'Closest FeI line (more than 0.05AA away) is a blend of this line.'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '5817.06'] += 'FeI or VI has too low loggf value.'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '5833.96'] += 'Two FeI lines have too low loggf values or there is a haluzinated line between.'
peak_inspection['classification'][peak_inspection['peak_wave_air'] == '6576.37'] = 'Unclear'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '6576.37'] += 'Only small residual peak and skyline 6577.63AA nearby.'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '6639.79'] += 'Both NiI and blended FeI have too high loggf value.'
peak_inspection['classification'][peak_inspection['peak_wave_air'] == '6648.11'] = 'Unclear'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '6648.11'] += 'Closest FeI has too low loggf value and is blended by unidentified line.'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '7714.26'] += 'Redder FeI line at 7714.6AA has too low loggf value, too.'
peak_inspection['classification'][peak_inspection['peak_wave_air'] == '7722.57'] = 'Unclear'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '7722.57'] += 'Closest MgI line seems to be blended by unidentified line.'
peak_inspection['classification'][peak_inspection['peak_wave_air'] == '7759.25'] = 'Unclear'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '7759.25'] += 'Closest MgI line seems to be blended by unidentified line.'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '7780.56'] += 'FeI line with too low loggf, blended by skyline.'
peak_inspection['classification'][peak_inspection['peak_wave_air'] == '7788.87'] = 'Unclear'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '7788.87'] += 'Closest NiI line seems to be blended by unidentified line.'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '7869.64'] += 'Both FeI and CoI lines have too low loggf values.'
peak_inspection['classification'][peak_inspection['peak_wave_air'] == '7788.87'] = 'Unclear'
peak_inspection['comments'][peak_inspection['peak_wave_air'] == '7788.87'] += 'Unidentified line.'
# peak_inspection['classification'][peak_inspection['peak_wave_air'] == '7840.83'] = 'Unclear'
# peak_inspection['comments'][peak_inspection['peak_wave_air'] == '7840.83'] += 'Haluzinated line in DR4? Blended with skyline'

for unique_classification in np.unique(peak_inspection['classification']):
    count = np.sum(peak_inspection['classification'] == unique_classification)
    print(unique_classification, ':', count, 'peaks')

peak_inspection.write('data/peak_inspection_table.fits', overwrite=True)
# peak_inspection.pprint_all()

In [ ]:
# To Be Confirmed:)
to_confirm = [
4719.17,
4722.89,
# 4727.26, # 4727.26  loggf --> too low  38 vs. 1 -> 37
4727.82,
# 4728.18 , ISM feature with width  10.6 Deeper  34 Shallower 0  ISM feature at  4728.28  (Vogroncic+2023 DIB catalogue or KI interstellar line)
4729.52,
4730.07,
4730.99, # Unclear, closest Fe I line (more than 0.05 Å away) is a blend of this line
# 4731.45 , loggf too low for  Fe II 34 vs. 1 -> 33
4732.23,
4733.80,
4734.16,
4735.36,
4737.66,
4738.30,
4738.81,
4739.59,
4740.47,
4740.93,
# 4741.52 , loggf too high for  Fe I 23 vs. 0 -> 23
4742.31,
4743.32,
4743.78,
4744.65,
4744.97,
4746.31,
4748.15,
# 4751.09 , loggf too low for  Fe I 31 vs. 0 -> 31
4752.20,
4754.22,
4754.59,
4755.74,
4756.29,
4758.68,
# 4759.28 , loggf too high for  Ti I 5 vs. 0 -> 5
4761.12,
4762.32,
4765.35,
# 4766.87 , loggf too high for  Fe I 9 vs. 0 -> 9
# 4767.84 , loggf too low for  Cr I 16 vs. 0 -> 16
4769.12,
4769.91,
4771.47,
# 4772.80 , loggf too high for  Fe I 30 vs. 0 -> 30
# 4773.95 , loggf too low for  Ce II 11 vs. 0 -> 11
4775.84,
# 4776.48 , loggf too low for  Fe I 33 vs. 0 -> 33
4779.43,
4783.20,
4783.98,
4787.29,
4789.59,
4791.16,
# 4792.31 , loggf too high for  Si I 72 vs. 0 -> 72
4796.26, # Unclear, blend with unidentified line, closest Co I (more than 0.05 Å away) has too low loggf
# 4798.24 , loggf too high for  Fe I 15 vs. 0 -> 15
4799.85, # Unclear, blend with unidentified line, closest Ti I (more than 0.05 Å away) has too low loggf
# 4800.13 , loggf too high for  Fe I 43 vs. 0 -> 43
4803.03,
# 4804.59 , loggf too low for  Fe I 13 vs. 1 -> 12
4805.00, # Unclear, blend with unidentified line, closest Ti II (more than 0.05 Å away)
# 4806.34 , loggf too high for  Ti II 12 vs. 0 -> 12
# 4806.98 , loggf too low for  Ni I 30 vs. 0 -> 30; both NiI and CeII have too low loggf value
4808.64, # CeII loggf too low
4809.10,
4809.93, # FeI loggf too high
4812.36, # Unclear, closest CrII is blended with unidentified line
4816.69,
4818.02,
4819.31,
4820.37,
# 4821.01 , loggf too high for  Fe I 7 vs. 0 -> 7
4822.35, # FeI loggf too high
4823.04,
4824.92,
# 4825.34 , # known missing; loggf too low for  Fe I 49 vs. 0 -> 49
4827.59,
# 4829.02 , loggf too high for  Ni I 10 vs. 0 -> 10
4831.41,
4831.87,
4833.89,
4834.58,
# 4836.84 , loggf too high for  Ni I 19 vs. 0 -> 19
# 4841.67 , loggf too high for  Fe I 15 vs. 0 -> 15
4844.52,
# 4846.40 , loggf too high for  Cr I 46 vs. 0 -> 46
4849.44,
# 4851.46 , loggf too low for  Cr I 16 vs. 0 -> 16
4853.40,
4853.90,
4854.64,
4855.19, # ISM feature
4856.25,
4858.13,
4859.84,
# 4861.49, Hbeta
4862.64,
4864.25,
4865.54,
4866.87,
4868.30,
4870.97,
4873.08,
4876.40,
4877.87,
4881.55,
4883.99,
# 4884.91 , loggf too low for  Cr I 36 vs. 0 -> 36
4887.76, # ZrI too high loggf
# 4889.00 , loggf too high for  Fe I 42 vs. 0 -> 42
4890.66,
# 4891.16 , loggf too high for  Fe I 60 vs. 0 -> 60
5657.77,
# 5658.65 , loggf too high for  Fe I 20 vs. 0 -> 20
5659.63,
# 5660.67 , loggf too low for  Si I 10 vs. 0 -> 10
5661.05,
5662.26,
# 5662.97 , loggf too high for  Fe I 11 vs. 0 -> 11
5663.95,
5665.59,
5667.07,
5667.67,
5671.50,
5671.94,
5673.47,
5675.66,
5676.59,
# 5677.68 , loggf too high for  Fe I 2 vs. 0 -> 2
5678.56,
# 5679.00 , loggf too low for  Fe I 2 vs. 0 -> 2
5681.13,
5682.33,
5683.10,
5684.08,
5684.58,
5686.27,
5686.93,
5687.42,
# 5688.19 , loggf too low for  Na I 10 vs. 0 -> 10
5688.62,
5690.54,
5691.52,
5693.66,
5695.02,
5696.23,
5698.14,
5698.63,
5700.27,
5701.31,
5702.41,
# 5702.95 , loggf too low for  Fe I 6 vs. 0 -> 6
# 5704.71 , loggf too low for  Fe I 17 vs. 0 -> 17
5706.07,
5706.67,
# 5708.10 , loggf too low for  Fe I 6 vs. 0 -> 6
5708.53,
5709.46,
# 5711.05 , loggf too low for  Mg I 13 vs. 0 -> 13
# 5712.64 , # known missing; loggf too low for  Cr I 10 vs. 0 -> 10
# 5714.17 , # known missing; loggf too low for  Fe I 41 vs. 0 -> 41
5715.21,
5717.78,
5723.74,
5727.19,
# 5730.85 , loggf too high for  Fe I 7 vs. 0 -> 7
5731.23,
5731.73,
# 5734.57 , loggf too high for  Fe I 4 vs. 0 -> 4
5735.67,
# 5738.24 , loggf too low for  Fe I 10 vs. 0 -> 10
# 5743.00 , loggf too low for  Fe I 4 vs. 0 -> 4
5743.38,
# 5747.81 , ISM Feature with width  8.4 Deeper  15 Shallower 0  ISM feature at  5747.62  (Vogroncic+2023 DIB catalogue or KI interstellar line)
5752.02,
5752.90,
5753.66,
# 5754.21 , loggf too high for  Si I 84 vs. 0 -> 84
# 5754.65 , loggf too low for  Ni I 23 vs. 0 -> 23
# 5755.36 , loggf too high for  Fe I 27 vs. 0 -> 27
5756.83, # FeI too low loggf
5757.60,
5758.48,
5759.51,
5760.12,
5761.43,
5762.52,
# 5762.91 , FeI to the left has too low loggf for  Fe I 35 vs. 0 -> 35
5771.60,
5774.23,
5779.64,
5780.30,
# 5780.85 , ISM Feature with width  28.6 Deeper  26 Shallower 0  ISM feature at  5780.59  (Vogroncic+2023 DIB catalogue or KI interstellar line)
5781.83,
# 5783.91 , loggf too low for  Fe I 10 vs. 0 -> 10
5785.22,
5785.55 , # Unclear. Neighbouring Mg line is too far away. loggf too high for  Mg I 7 vs. 0 -> 7
5787.30,
5790.15,
5790.80,
5796.11, # NiI too low loggf
5796.77,
# 5797.48 , with width  14.8 Deeper  2 Shallower 8  ISM feature at  5797.19  (Vogroncic+2023 DIB catalogue or KI interstellar line)
5797.91,
5806.28,
# 5809.29 , ISM Feature with width  6.7 Deeper  6 Shallower 0  ISM feature at  5809.42  (Vogroncic+2023 DIB catalogue or KI interstellar line)
5810.49,
5812.79,
# 5815.64 , loggf too high for  Fe I 6 vs. 0 -> 6
5816.29, # Unidentified line , loggf too low for  Fe I 30 vs. 0 -> 30
# 5817.06 , loggf too low for  Fe I 10 vs. 0 -> 10
# 5822.42 , # known missing; loggf too low for  Cr I 14 vs. 0 -> 14
# 5827.45 , loggf too high for  Fe I 9 vs. 0 -> 9
# 5827.89 , loggf too low for  Fe I 9 vs. 0 -> 9
5831.06, # FeI loggf too high
# 5831.94 , loggf too high for  Ti I 8 vs. 0 -> 8
# 5833.96 , loggf too high for  Fe I 50 vs. 0 -> 50
# 5834.51 , loggf too high for  Ti I 16 vs. 0 -> 16
# 5835.11 , loggf too low for  Fe I 11 vs. 0 -> 11
# 5837.95 , loggf too high for  Ti I 11 vs. 0 -> 11
5839.60,
# 5841.18 , loggf too high for  Ti I 23 vs. 0 -> 23
# 5845.28 , loggf too low for  Fe I 5 vs. 0 -> 5
5846.10,
5848.02,
5857.48,
5858.30,
# 5859.23 , loggf too high for  Fe I 16 vs. 0 -> 16
5862.19,
# 6494.47 , loggf too low for  Fe I 20 vs. 0 -> 20
# 6495.03 , loggf too low for  Fe I 21 vs. 1 -> 20
6497.24,
# 6497.68 , loggf too low for  Ti I 8 vs. 1 -> 7
# 6498.95 , loggf too low for  Fe I 13 vs. 1 -> 12
6499.58, # CaI loggf too low
6501.66,
6504.12,
# 6509.61 , loggf too high for  Fe I 5 vs. 0 -> 5
6514.78,
6516.11,
# 6518.76 , loggf too low for  Si I 40 vs. 0 -> 40
6520.15,
6522.10,
# 6526.71 , loggf too low for  Si I 9 vs. 0 -> 9
# 6528.54 , # known missing; loggf too low for  Fe I 39 vs. 0 -> 39
6530.05,
6533.40,
# 6533.90 , loggf too low for  Fe I 18 vs. 0 -> 18
6539.01,
6541.48,
6546.02,
6547.60,
6553.28,
6555.80,
6558.07,
6560.28,
6561.41,
# 6563.12, Balmer line Halpha
6564.63,
6565.64,
6566.34,
6567.03,
6567.66,
6568.99,
6572.52,
6574.79,
6576.37, # Unclear. Not very strong... Potentially neighbouring Skyline? 6576.37 , loggf too high for  Ni I 4 vs. 1 -> 3
6577.63,
6579.97,
6584.45,
# 6592.65 , ISM Feature with width  11.1 Deeper  0 Shallower 64  ISM feature at  6592.53  (Vogroncic+2023 DIB catalogue or KI interstellar line)
6593.91,
6595.74,
6596.50,
6599.15,
6602.49,
6609.75,
# 6613.41 , ISM Feature with width  6.2 Deeper  33 Shallower 0  ISM feature at  6613.66  (Vogroncic+2023 DIB catalogue or KI interstellar line)
6618.52,
6623.82,
6624.39,
6625.52,
# 6627.54 , loggf too low for  Fe I 17 vs. 0 -> 17
6631.96,
6633.35,
# 6634.11 , loggf too low for  Fe I 40 vs. 0 -> 40
6635.05,
6638.14,
# 6639.79 , NiI and FeI loggf too low for  Ni I 17 vs. 0 -> 17
6642.56,
6643.57, # NiI logg too low or indeed a blend with unidentified line?
6644.27,
6645.21,
# 6646.92 , loggf too low for  Fe I 43 vs. 0 -> 43
6648.11 , # Actually unidentified blend. loggf too low for  Fe I 13 vs. 0 -> 13
6650.13,
6651.02,
6655.56,
6656.89,
6659.79,
6660.55,
6665.47,
# 6666.54 , loggf too high for  Ti I 31 vs. 0 -> 31
# 6668.37 , loggf too low for  Ti I 15 vs. 0 -> 15
6671.02,
6677.71,
6678.15,
6682.31,
6684.71,
6685.47,
6687.55,
6688.50,
# 6692.28 , loggf too high for  Fe I 31 vs. 0 -> 31
6694.49,
6696.76,
6697.33,
6703.20,
6703.71,
6704.59,
6705.22,
6707.81, # Li line.
6709.07,
# 6711.53 , loggf too high for  Ni I 17 vs. 0 -> 17
6712.60,
# 6713.04 , loggf too low for  Fe I 13 vs. 1 -> 12
# 6717.52 , loggf too low for  Fe I 60 vs. 0 -> 60
6719.61,
6724.09,
6726.42,
7690.08,
# 7691.55 , loggf too low for  Mg I 7 vs. 3 -> 4
# 7695.59 , loggf too low for  Si I 6 vs. 1 -> 5
# 7698.75 , ISM Feature with width  23.5 Deeper  4 Shallower 8  ISM feature at  7698.9643  (Vogroncic+2023 DIB catalogue or KI interstellar line)
7710.15,
# 7711.76, # Skyline
# 7712.72 , Skyline at  7712.646  (strength  8.7 )
# 7714.26 , loggf too low for  Ni I 43 vs. 1 -> 42
7715.22,
# 7716.62 , Skyline at  7716.909  (strength  10.8 )
# 7719.04 , loggf too low for  Fe I 10 vs. 1 -> 9
# 7722.57 , loggf too high for  Mg I 20 vs. 3 -> 17
# 7723.23 , loggf too low for  Fe I 11 vs. 1 -> 10
7727.42,
7732.34,
7742.49,
# 7748.29 , loggf too low for  Fe I 10 vs. 1 -> 9
# 7750.65 , Skyline at  7750.646  (strength  20.2 )
# 7751.09 , loggf too low for  Fe I 11 vs. 6 -> 5
# 7759.25 , loggf too high for  Mg I 30 vs. 1 -> 29
7771.96, # OI line...
7772.62,
7774.24, # OI line...
7775.42, # OI line...
# 7780.56 , Both loggf too low and skyline loggf too low for  Fe I 13 vs. 2 -> 11
7787.10,
# 7788.87 , loggf too low for  Ni I 37 vs. 1 -> 36
# 7793.94 , Skyline at  7794.084  (strength  14.5 )
# 7799.96 , loggf too low for  Si I 77 vs. 0 -> 77
# 7807.83 , loggf too low for  Fe I 6 vs. 2 -> 4
# 7808.49 , Skyline at  7808.49  (strength  6.7 )
7811.06,
7820.54,
# 7821.79 , Skyline at  7821.5  (strength  15.8 )
7826.79,
# 7832.23 , loggf too high for  Fe I 14 vs. 0 -> 14
7835.17, # AlI line; assymetric?
7835.98, # AlI line; assymetric?
7840.83, # 7840.83 , Skyline at  7841.271  (strength  6.0 )
# 7842.15 , loggf too low for  Ti I 2 vs. 2 -> 0
7843.04,
# 7844.51 , loggf too low for  Fe I 3 vs. 1 -> 2
7845.31,
7848.62,
# 7853.25 , Skyline at  7853.399  (strength  11.5 )
7855.90,
# 7860.90 , Skyline at  7860.675  (strength  6.4 )
# 7869.64 , loggf too low for  Fe I 11 vs. 2 -> 9
# 7870.60 , Skyline at  7870.745  (strength  7.2 )
7871.48,
7872.73
]

In [ ]:
# Now make a LaTeX table out of that

latex_table = r'''\begin{tabular}{ccccccccccc}
\hline
Peak & $\lambda_\mathrm{air}$ & $s_\lambda$ & Width  & Classification & Shallower & Hinkle+(2000) & Nearest DR4 & $\log gf$ & Sky              & Comments \\
     & $\text{\AA}$           &             & pixels &                & Deeper    & Species       & Species     & \textsc{DEPTH} & $f_\mathrm{sky}$ &          \\
\hline
'''

loggf_reference_note = []

for row in peak_inspection:
    if row['dr4_log_gf_ref'] != 'nan':
        linelist_reference = row['dr4_log_gf_ref'].replace(' ','').replace('&',r'\&').replace('_',r'\_')
        bibtex_reference = r'\citet{'+row['dr4_log_gf_ref'].replace(' ','')+'}'
        
        if linelist_reference != '--':
            loggf_reference_note.append(rf"{linelist_reference}: {bibtex_reference}")

    if row['peak_wave_air'] in ['4727.26', '7799.96', '4731.45', '4861.49', '5780.85', '7853.25']:
        
        # Nearest Hinkle+2000 line info
        hinkle_info1 = f'{row['hinkle_species']} ({row['hinkle_distance']:+.2f} Å) &'
        hinkle_info2 = f'({row['hinkle_wave_air']:.2f} Å) & '
        if row['hinkle_species'] == 'None':
            hinkle_info1 = 'None &'
            hinkle_info2 = ' & '

        # Nearest GALAH DR4 line info
        dr4_info1 = f'{row['dr4_line_name']} ({row['dr4_distance']:+.2f} Å) & {row['dr4_log_gf']:.2f} &'
        dr4_info2 = f'({row['dr4_wave_air']:.2f} Å) & ({linelist_reference}) & {row['dr4_depth']:.2f} &'
        if row['dr4_line_name'] == 'None':
            dr4_info1 = 'None &  &  &'
            dr4_info2 = 'None &  &  &'

        # Skyline info
        if np.isfinite(row['skyline_strength']):
            skyline_info = f'{row['skyline_strength']:.1f} &'
        else:
            skyline_info = ' &'

        # first row
        latex_table += rf"{row['peak_index']} & {row['peak_wave_air']} & {row['significance']:.1f} & {row['width']:.0f} & {row['classification']} & +{row['percent_shallower']}\% & {hinkle_info1} {dr4_info1} {skyline_info} "

        if row['peak_wave_air'] == '4727.26': latex_table += r'Fig.~\ref{fig:example_significant_residual_regions}A'
        if row['peak_wave_air'] == '7799.96': latex_table += r'Fig.~\ref{fig:example_significant_residual_regions}B'
        if row['peak_wave_air'] == '4731.45': latex_table += r'Fig.~\ref{fig:example_significant_residual_regions}C'
        if row['peak_wave_air'] == '4861.49': latex_table += r'Fig.~\ref{fig:example_significant_residual_regions}D'
        if row['peak_wave_air'] == '5780.85': latex_table += r'Fig.~\ref{fig:example_significant_residual_regions}E'
        if row['peak_wave_air'] == '7853.25': latex_table += r'Fig.~\ref{fig:example_significant_residual_regions}F'

        latex_table += " \\\\\n"
        # second row
        latex_table += rf"  &         &     &              & -{row['percent_deeper']}\% & {hinkle_info2} {dr4_info2} "+" \\\\\n"

latex_table += r'''\dots & \dots & \dots & \dots & \dots & \dots & \dots & \dots & \dots & \dots \\
\hline
\end{tabular}
'''
joined_references = ", \n".join(set(loggf_reference_note))
joined_references = joined_references.replace(r'\citet{KL-astro}', r'GALAH manually adjusted')
joined_references = joined_references.replace(r'\citet{MA-astro}', r'GALAH manually adjusted')
joined_references = joined_references.replace(r'\citet{GESB82c+BWL}', r'\citet{GESB82c,BWL}')
joined_references = joined_references.replace(r'\citet{BK+BWL}', r'\citet{BK,BWL}')
joined_references = joined_references.replace(r'\citet{GESB86+BWL}', r'\citet{GESB86,BWL}')
joined_references = joined_references.replace(r'\citet{2014ApJS..215...20L_1985A&A...153..109W}', r'\citet{2014ApJS..215...20L,1985A&A...153..109W}')

joined_references = (joined_references).replace('|',',')
latex_table += r'\footnotesize{$\log gf$ references: '+joined_references+'}'

np.savetxt('tex_text/tabular_peak_information.tex', [latex_table], fmt='%s')
print(latex_table)

In [ ]:
"""
# Check for nearby lines in Hinkle+ 2000 within +-1 pixel of each significant peak, and classify the peaks accordingly
classification = dict()
classification['residual_peak_index'] = []
classification['residual_significance'] = []
classification['residual_wave_air'] = []
classification['ref_line_wave_air'] = []
classification['ref_line_distance_lambda'] = []
classification['ref_line_distance_pixels'] = []
classification['ref_line_source'] = []
classification['ref_line_species'] = []
classification['residual_classification'] = []
classification['classification_comments'] = []

significant_peaks_table['wave'] = np.round(significant_peaks_table['wave'],3)

for peak_index, significant_peak in enumerate(significant_peaks_table):

    classification['residual_peak_index'].append(peak_index)
    classification['residual_wave_air'].append(np.round(significant_peak['wave'],3))
    classification['residual_significance'].append(np.round(significant_peak['significance'],1))

    # within +-1 pixel, find the closest line in Hinkle+ 2000
    if (significant_peak['wave'] > 4719.0299) & (significant_peak['wave'] < 4893.8808):
        pixelscale = 0.0460
    elif (significant_peak['wave'] > 5655.5298) & (significant_peak['wave'] < 5863.0182):
        pixelscale = 0.0547
    elif (significant_peak['wave'] > 6486.0104) & (significant_peak['wave'] < 6726.4572):
        pixelscale = 0.0631
    elif (significant_peak['wave'] > 7689.1261) & (significant_peak['wave'] < 7874.3127):
        pixelscale = 0.0735

    window = 2 * pixelscale

    nearby_lines = hinkle_lines_table[np.abs(hinkle_lines_table['wave_air'] - significant_peak['wave']) < window]
    if len(nearby_lines) > 0:
        closest_line_index = np.argmin(np.abs(nearby_lines['wave_air'] - significant_peak['wave']))
        closest_line = nearby_lines[closest_line_index]

        classification['ref_line_species'].append(closest_line['species'])
        classification['ref_line_wave_air'].append(closest_line['wave_air'])
        classification['ref_line_distance_lambda'].append(np.round(np.abs(closest_line['wave_air'] - significant_peak['wave']),3))
        classification['ref_line_distance_pixels'].append(np.round(np.abs(closest_line['wave_air'] - significant_peak['wave'])/pixelscale,1))
        classification['ref_line_source'].append('Hinkle+2000')
        if closest_line['species'] == 'FeI':
            classification['residual_classification'].append('FeI Line')
        else:
            classification['residual_classification'].append('Other Line')
        classification['classification_comments'].append('Automatic classification')
    elif significant_peak['wave'] in [4861.492, 6563.119]:
        if significant_peak['wave'] == 4861.492:
            hinkle_species = 'HI'
            hinkle_line = 4861.32
            classification['residual_classification'].append(r'$\mathrm{H_\beta}$')
        elif significant_peak['wave'] == 6563.119:
            hinkle_species = 'HI'
            hinkle_line = 6562.819
            classification['residual_classification'].append(r'$\mathrm{H_\alpha}$')

        classification['ref_line_species'].append(hinkle_species)
        classification['ref_line_wave_air'].append(hinkle_line)
        classification['ref_line_distance_lambda'].append(np.round(np.abs(hinkle_line - significant_peak['wave']),3))
        classification['ref_line_distance_pixels'].append(np.round(np.abs(hinkle_line - significant_peak['wave'])/pixelscale,1))
        classification['ref_line_source'].append('Hinkle+2000')
        classification['classification_comments'].append('--')
    else:
        classification['ref_line_species'].append('None')
        classification['ref_line_wave_air'].append(np.nan)
        classification['ref_line_distance_lambda'].append(np.nan)
        classification['ref_line_distance_pixels'].append(np.nan)
        classification['ref_line_source'].append('--')
        classification['residual_classification'].append('--')
        classification['classification_comments'].append('--')

for key in ['residual_wave_air','ref_line_wave_air']:
    classification[key] = np.array(classification[key])
"""

In [ ]:
"""
sky_line_matches = np.array([
    # [6498.757, 6494.466], #?
    # [6522.419, 6522.104], #?
    # [6533.020, 6533.903], #?
    # [6544.062, ],
    [6553.654, 6553.275],
    [6568.798, 6568.987],
    [6577.316, 6577.632],
    # [6596.625, ],
    # [6604.007, ],
    # [6604.197, ],
    # [7712.279, ],
    [7712.646, 7712.720],
    [7715.072, 7714.263],
    # [7715.880, ],
    # [7716.909, ],
    [7718.085, 7719.041],
    [7723.671, 7722.569],
    [7726.023, 7727.42],
    [7749.396, 7748.294],
    [7750.646, 7751.087],
    [7759.980, 7759.245],
    # [7773.357, 7774.239], # <- actually OI triplet
    [7780.413, 7780.56],
    [7794.084, 7793.937],
    [7808.490, 7807.829],
    [7821.500, 7821.794],
    [7841.271, 7840.830],
    [7853.399, 7853.252],
    [7860.675, 7860.896],
    [7867.731, 7869.642],
    [7870.745, 7871.480]
])

for sky_line_match in sky_line_matches:
    index_in_classification = np.where(classification['residual_wave_air'] == sky_line_match[1])[0]
    if len(index_in_classification) > 0:
        index_in_classification = index_in_classification[0]
        classification['ref_line_species'][index_in_classification] = 'Sky'
        classification['ref_line_wave_air'][index_in_classification] = sky_line_match[0]
        classification['ref_line_distance_lambda'][index_in_classification] = np.round(np.abs(sky_line_match[0] - sky_line_match[1]),3)
        classification['ref_line_distance_pixels'][index_in_classification] = np.round(np.abs(sky_line_match[0] - sky_line_match[1])/pixelscale,1)
        classification['ref_line_source'][index_in_classification] = 'Sky Obs.'
        classification['residual_classification'][index_in_classification] = 'Sky Line'

        find_strength = sky_lines['strength'][sky_lines['wave'] == sky_line_match[0]]

        classification['classification_comments'][index_in_classification] = 'Sky line strength '+str(find_strength[0])
"""

In [ ]:
"""
# Guided identifications via comparison of Hinkle+2000 & Linelist

manual_classifications = np.array([
    # lambda , lambda   , source   , LOGG, SYN_, species  , Classif., comments
    # Resid.   DR4      ,            FLAG, FLAG
    # > 5 Sigma
    [4748.148, 4748.1760, 'DR4/K07', '?', '-', 'MnI'    , 'Manual'            , r'Visible but unidentified in Hinkle+2000'],
    [4771.47 , 4771.4503, 'DR4/K09', '?', '-', 'ScI/CrI', 'Manual'            , r'Visible but unidentified in Hinkle+2000'],
    [5693.656, 5693.6162, 'DR4/K07', 'N', 'U', 'FeI'    , 'FeI Line'          , r'LOGF\_FLAG=N, SYN\_FLAG=U in linelist'],
    [5754.209, 5754.4025, 'DR4/K07', '?', 'N', 'FeI'    , 'FeI Line'          , r'SYN\_FLAG=N in linelist'], # <- SiI is mismatch with automatic comparison to Hinkle+2000
    [5667.592, 5667.5940, 'DR4/K10', '-', '-', 'TiI'    , 'Fake Synthesis'    , r'Wrong $\log gf$. No TiI here (actually FeI)'],
    [7799.964, 7799.9957, 'DR4/K07', '-', '-', 'SiI'    , 'Missing Synthesis' , r'Too low $\log gf$'],
    # 4 < Sigma < 5
    [5831.062, 5831.0567, 'DR4/K07', '-', '-', 'FeI'    , 'Fake Synthesis'    , r'Wrong $\log gf$. No FeI here (no line visible in Hinkle+2000)'],
    # Unidentified
    [7732.344, np.nan,    '?'      , '-', '-', '?'      , 'Unidentified'      , r'Unidentified shallow, broad stellar feature (correlates with [Fe/H] in M3, visible in Hinkle+2000).'],  
])

for manual_classification in manual_classifications:
    index_in_classification = np.where(classification['residual_wave_air'] == float(manual_classification[0]))[0]
    if len(index_in_classification) > 0:
        index_in_classification = index_in_classification[0]

        classification['ref_line_species'][index_in_classification] = manual_classification[5]
        classification['ref_line_wave_air'][index_in_classification] = float(manual_classification[1])
        classification['ref_line_distance_lambda'][index_in_classification] = np.round(np.abs(float(manual_classification[1]) - classification['residual_wave_air'][index_in_classification]),3)
        classification['ref_line_distance_pixels'][index_in_classification] = np.round(np.abs(float(manual_classification[1]) - classification['residual_wave_air'][index_in_classification])/pixelscale,1)
        classification['ref_line_source'][index_in_classification] = manual_classification[2]
        classification['residual_classification'][index_in_classification] = manual_classification[6]
        classification['classification_comments'][index_in_classification] = manual_classification[7]

"""

In [ ]:
"""

# # Create and save table as FITS file and show first few lines
# classified = Table(classification)
# classified.write('data/significant_peaks_classification.fits', overwrite=True)

classified = Table.read('data/significant_peaks_classification.fits')
classified.sort('residual_significance', reverse=True)
classified[:5]

"""

In [ ]:
"""

# Create LaTeX overview table of unique classifications for paper.

unique_classifications, unique_classifications_indices, unique_classifications_counts = np.unique(classified['residual_classification'], return_counts=True, return_index=True)

unique_classifications_sorted = classified[unique_classifications_indices]
unique_classifications_sorted.sort('residual_significance', reverse=True)

latex_table = r'''\begin{tabular}{cccccccccp{4.75cm}}
\hline
Lines in & Classification & Largest & Residual                            & Reference                           &                  &            & Reference & Reference & Comments \\
Class.   &                & Signif. & $\lambda_\mathrm{air}~/~\text{\AA}$ & $\lambda_\mathrm{air}~/~\text{\AA}$ & $\Delta \lambda$ & $\Delta$px & Source    & Species   &          \\
\hline
'''
for row in unique_classifications_sorted:
    latex_table += f"{unique_classifications_counts[row['residual_classification'] == unique_classifications][0]} & {row['residual_classification']} & {row['residual_significance']:.1f} & {row['residual_wave_air']:.3f} & "
    if np.isnan(row['ref_line_wave_air']):
        latex_table += "-- & "
    else:
        latex_table += f"{row['ref_line_wave_air']:.3f} & "
    if np.isnan(row['ref_line_distance_lambda']):
        latex_table += "-- & -- & "
    else:
        latex_table += f"{row['ref_line_distance_lambda']:.3f} & {row['ref_line_distance_pixels']:.1f} & "
    latex_table += f"{row['ref_line_source']} & "
    latex_table += f"{row['ref_line_species']} & {row['classification_comments']} \\\\\n"
latex_table += r'''\dots & \dots & \dots & \dots & \dots & \dots & \dots & \dots & \dots & \dots \\
\hline
\end{tabular}'''
np.savetxt('tex_text/tabular_significant_peaks_classification.tex', [latex_table], fmt='%s')
print(latex_table)

"""

## 3.2 Residual Cannon Models

In [ ]:
cannon_models = dict()

tests_run = [1,2,3,4,5,6,7]

# Populate the relevant dictionary of input data
for test in tests_run:

    if test in [1,5,6]: data_to_use = model1_data
    if test in [2,7]: data_to_use = model2_data
    if test == 3: data_to_use = model3_data
    if test == 4: data_to_use = model4_data

    label_names = list(data_to_use['labels'].dtype.names[1:])

    # Test 7: Run 3-label Cannon on abundance sample
    if test == 7:
        label_names = ['teff','logg','fe_h']

    cannon_models['model_'+str(test)] = dict()
    cannon_models['model_'+str(test)]['label_names'] = label_names
    cannon_models['model_'+str(test)]['sobject_id'] = data_to_use['labels']['sobject_id']
    cannon_models['model_'+str(test)]['labels'] = np.array([data_to_use['labels'][label] for label in cannon_models['model_'+str(test)]['label_names']]).T

    # For tests 5 and 6, we use the 6% (usually masked) and 94% (unmasked) parts of the spectrum, respectively.
    # For the other tests, we use the full wavelength range
    if test == 5:
        cannon_models['model_'+str(test)]['flux']  = data_to_use['flux'][:,~mask_common_wavelength]
        cannon_models['model_'+str(test)]['ivar']  = data_to_use['ivar'][:,~mask_common_wavelength]
        cannon_models['model_'+str(test)]['wave'] = common_wavelength[~mask_common_wavelength]
    elif test == 6:
        cannon_models['model_'+str(test)]['flux'] = data_to_use['flux'][:,mask_common_wavelength]
        cannon_models['model_'+str(test)]['ivar'] = data_to_use['ivar'][:,mask_common_wavelength]
        cannon_models['model_'+str(test)]['wave'] = common_wavelength[mask_common_wavelength]
    else:
        cannon_models['model_'+str(test)]['flux']  = data_to_use['flux']
        cannon_models['model_'+str(test)]['ivar']  = data_to_use['ivar']
        cannon_models['model_'+str(test)]['wave'] = common_wavelength

    if rerun_everything:

        # Set up the model
        cannon_model = tc.CannonModel(
            training_set_labels = cannon_models['model_'+str(test)]['labels'],
            training_set_flux = cannon_models['model_'+str(test)]['flux'],
            training_set_ivar = cannon_models['model_'+str(test)]['ivar'],
            vectorizer=tc.vectorizer.PolynomialVectorizer(list(cannon_models['model_'+str(test)]['label_names']), 2),dispersion=cannon_models['model_'+str(test)]['wave'])

        # Train the model
        cannon_models['model_'+str(test)]['theta'], cannon_models['model_'+str(test)]['s2'], cannon_models['model_'+str(test)]['metadata'] = cannon_model.train()
        cannon_models['model_'+str(test)]['model'] = cannon_model

        # Save the model
        cannon_model.write('models/cannon_model_m'+str(test)+'.model', overwrite=True)
            
        # Save the theta coefficients in a FITS file for easy plotting
        thetas = Table()
        thetas['wave'] = cannon_models['model_'+str(test)]['wave']
        for theta_index in range(np.shape(cannon_models['model_'+str(test)]['theta'])[1]):
            keyword = 'df_d'+cannon_models['model_'+str(test)]['model'].vectorizer.get_human_readable_label_term(theta_index)
            if theta_index == 0:
                keyword = 'f_0'
            thetas[keyword] = cannon_models['model_'+str(test)]['theta'][:,theta_index]
        thetas.write('models/cannon_model_m'+str(test)+'_thetas.fits', overwrite=True)

        # Test the model
        cannon_models['model_'+str(test)]['out'], cannon_models['model_'+str(test)]['cov'], dummy = cannon_model.test(cannon_models['model_'+str(test)]['flux'], cannon_models['model_'+str(test)]['ivar'], threads=1)
        cannon_models['model_'+str(test)]['flux_out'] = np.array(cannon_model.__call__(cannon_models['model_'+str(test)]['out']))

        np.savez('models/cannon_model_m'+str(test)+'_recovery.npz', out=cannon_models['model_'+str(test)]['out'], cov=cannon_models['model_'+str(test)]['cov'], flux_out=cannon_models['model_'+str(test)]['flux_out'], sobject_id=cannon_models['model_'+str(test)]['sobject_id'])

    # Read in the trained model and recovery results
    cannon_model = tc.CannonModel.read('models/cannon_model_m'+str(test)+'.model')
    recovery = np.load('models/cannon_model_m'+str(test)+'_recovery.npz')
    cannon_models['model_'+str(test)]['theta'] = cannon_model.theta
    cannon_models['model_'+str(test)]['s2'] = cannon_model.s2
    cannon_models['model_'+str(test)]['model'] = cannon_model
    cannon_models['model_'+str(test)]['out'] = recovery['out']
    cannon_models['model_'+str(test)]['cov'] = recovery['cov']
    cannon_models['model_'+str(test)]['flux_out'] = recovery['flux_out']

    # Save the theta coefficients in a FITS file for easy plotting
    thetas = Table()
    thetas['wave'] = cannon_models['model_'+str(test)]['wave']
    for theta_index in range(np.shape(cannon_models['model_'+str(test)]['theta'])[1]):
        keyword = 'df_d'+cannon_models['model_'+str(test)]['model'].vectorizer.get_human_readable_label_term(theta_index)
        if theta_index == 0:
            keyword = 'f_0'
        thetas[keyword] = cannon_models['model_'+str(test)]['theta'][:,theta_index]
    thetas.write('models/cannon_model_m'+str(test)+'_thetas.fits', overwrite=True)


In [ ]:
# Visualize the recovery of labels for each model, using different colormaps for the different versions
for test in tests_run:

    if test == 1: cmap = 'Greys_r'
    if test == 2: cmap = 'Blues_r'
    if test == 3: cmap = 'Oranges_r'
    if test == 4: cmap = 'Purples_r'
    if test == 5: cmap = 'Reds_r'
    if test == 6: cmap = 'RdPu_r'
    if test == 7: cmap = 'GnBu_r'; color = 'deepskyblue'

    for index, label in enumerate(cannon_models['model_'+str(test)]['label_names']):

        f, ax = plt.subplots(figsize=(4,3))
        
        recovery_data = cannon_models['model_'+str(test)]['out'][:,index] - cannon_models['model_'+str(test)]['labels'][:,index]

        if label == 'teff':
            recovery_data = recovery_data.clip(min=-2000,max=2000)
        elif label == 'logg':
            recovery_data = recovery_data.clip(min=-1.9,max=1.9)
        elif label == 'ba_fe':
            recovery_data = recovery_data.clip(min=-1.,max=1.)
        elif label == 'a_ks':
            recovery_data = recovery_data.clip(min=-0.15,max=0.15)

        input_minmax = [np.min(cannon_models['model_'+str(test)]['labels'][:,index]), np.max(cannon_models['model_'+str(test)]['labels'][:,index])]
        input_minmax = [1.1*input_minmax[0]-0.1*input_minmax[1],1.1*input_minmax[1]-0.1*input_minmax[0]]
        delta_absmax = np.max(np.abs(np.min(recovery_data)))
        delta_minmax = [-1.1*delta_absmax, 1.1*delta_absmax]

        ax.hist2d(
            cannon_models['model_'+str(test)]['labels'][:,index],
            recovery_data,
            cmin = 1,
            cmap = cmap,
            bins = (np.linspace(input_minmax[0],input_minmax[1],100),np.linspace(delta_minmax[0],delta_minmax[1],100)),
            norm = LogNorm()
        )

        if label == 'teff':
            fancy_label = r'$T_\mathrm{eff}~/~\mathrm{K}$'
        elif label == 'logg':
            fancy_label = r'$\log (g~/~\mathrm{cm\,s^{-2}})$'
        elif label == 'fe_h':
            fancy_label = r'$\mathrm{[Fe/H]}$'
        elif label == 'ebv':
            fancy_label = r'$E(B-V)~/~\mathrm{mag}$'
        elif label == 'a_ks':
            fancy_label = r'$A_\mathrm{K_S}~/~\mathrm{mag}$'
        elif label == 'na_fe':
            fancy_label = r'$\mathrm{[Na/Fe]}$'
        elif label == 'mg_fe':
            fancy_label = r'$\mathrm{[Mg/Fe]}$'
        elif label == 'mn_fe':
            fancy_label = r'$\mathrm{[Mn/Fe]}$'
        elif label == 'ba_fe':
            fancy_label = r'$\mathrm{[Ba/Fe]}$'
        elif label == 'logRHK':
            fancy_label = r'$\log R^\prime_\mathrm{HK}$'
        else:
            fancy_label = label

        # Remember the input distribution with significant decimal places
        input_distribution = np.percentile(cannon_models['model_'+str(test)]['labels'][:,index], q=[16,50,84])
        if np.max([[input_distribution[1]-input_distribution[0]],[input_distribution[2]-input_distribution[1]]]) > 100:
            fmt = "{:.0f}"
            input_distribution = np.array(np.round(input_distribution,-1),dtype=int)
        elif np.max([[input_distribution[1]-input_distribution[0]],[input_distribution[2]-input_distribution[1]]]) > 10:
            fmt = "{:.0f}"
            input_distribution = np.array(np.round(input_distribution,0),dtype=int)
        elif np.max([[input_distribution[1]-input_distribution[0]],[input_distribution[2]-input_distribution[1]]]) > 1:
            fmt = "{:.1f}"
            input_distribution = np.round(input_distribution,1)
        elif np.max([[input_distribution[1]-input_distribution[0]],[input_distribution[2]-input_distribution[1]]]) > 0.1:
            fmt = "{:.2f}"
            input_distribution = np.round(input_distribution,2)
        else:
            fmt = "{:.3f}"
            input_distribution = np.round(input_distribution,3)

        ax.set_xlabel('Input '+fancy_label[:-1]+ r' \in '+fmt.format(input_distribution[1])+'_{-'+fmt.format(input_distribution[1]-input_distribution[0])+'}^{+'+fmt.format(input_distribution[2]-input_distribution[1])+'}$',fontsize=15)

        # Calculate the percentiles of the residuals with significant decimal places
        percentiles = np.percentile(cannon_models['model_'+str(test)]['out'][:,index] - cannon_models['model_'+str(test)]['labels'][:,index], q = [16,50,84])
        if np.max([[percentiles[1]-percentiles[0]],[percentiles[2]-percentiles[1]]]) > 100:
            fmt = "{:.0f}"
            percentiles = np.array(np.round(percentiles,-1),dtype=int)
        elif np.max([[percentiles[1]-percentiles[0]],[percentiles[2]-percentiles[1]]]) > 10:
            fmt = "{:.0f}"
            percentiles = np.array(np.round(percentiles,0),dtype=int)
        elif np.max([[percentiles[1]-percentiles[0]],[percentiles[2]-percentiles[1]]]) > 1:
            fmt = "{:.1f}"
            percentiles = np.round(percentiles,1)
        elif np.max([[percentiles[1]-percentiles[0]],[percentiles[2]-percentiles[1]]]) > 0.1:
            fmt = "{:.2f}"
            percentiles = np.round(percentiles,2)
        else:#if np.max([[percentiles[1]-percentiles[0]],[percentiles[2]-percentiles[1]]]) > 0.01:
            fmt = "{:.3f}"
            percentiles = np.round(percentiles,3)
        
        ax.set_ylabel(r'$\Delta$'+fancy_label,fontsize=15)
        ax.text(0.05,0.05,r'$\Delta = '+fmt.format(percentiles[1])+'_{-'+fmt.format(percentiles[1]-percentiles[0])+'}^{+'+fmt.format(percentiles[2]-percentiles[1])+'}$',va='bottom',ha='left',transform=ax.transAxes,fontsize=12)

        ax.axhline(0.0,c='k',lw=1,ls='dashed')

        ax.set_aspect('equal', adjustable='datalim')
        
        plt.tight_layout()
        plt.savefig('figures/recovery_model'+str(test)+'_'+label+'.png',dpi=200,bbox_inches='tight')
        plt.show()
        plt.close()

In [ ]:
# Plot linear coefficients for each model in one figure, using different colormaps for the different versions
for test in tests_run:

    if test == 1: cmap = 'Greys_r'; color = 'k'
    if test == 2: cmap = 'Blues_r'; color = 'C0'
    if test == 3: cmap = 'Oranges_r'; color = 'C1'
    if test == 4: cmap = 'Purples_r'; color = 'C4'
    if test == 5: cmap = 'Reds_r'; color = 'C3'
    if test == 6: cmap = 'RdPu_r'; color = 'C6'
    if test == 7: cmap = 'GnBu_r'; color = 'deepskyblue'

    rows = 2+len(cannon_models['model_'+str(test)]['label_names'])

    f, gs = plt.subplots(
        rows,
        4,
        figsize=(10,1*rows),
        sharey=True
    )
    gs = gs.flatten()

    for index in range(rows):

        if index == 0:
            ylabel = 'Median\n'+r'Model $F_\lambda$'
            labelfontsize = 12
        elif index == 1:
            ylabel = 'Model\nScatter'
            labelfontsize = 12
        else:
            label = cannon_models['model_'+str(test)]['label_names'][index-2]
            labelfontsize = 15
            if label == 'teff':
                fancy_label = r'T_\mathrm{eff}'#r'$T_\mathrm{eff}~/~\mathrm{K}$'
            elif label == 'logg':
                fancy_label = r'\log (g)'#~/~\mathrm{cm\,s^{-2}})$'
            elif label == 'fe_h':
                fancy_label = r'\mathrm{[Fe/H]}'
            elif label == 'na_fe':
                fancy_label = r'\mathrm{[Na/Fe]}'
            elif label == 'mg_fe':
                fancy_label = r'\mathrm{[Mg/Fe]}'
            elif label == 'mn_fe':
                fancy_label = r'\mathrm{[Mn/Fe]}'
            elif label == 'ba_fe':
                fancy_label = r'\mathrm{[Ba/Fe]}'
            elif label == 'ebv':
                fancy_label = r'E(B-V)'#~/~\mathrm{mag}$'
            elif label == 'a_ks':
                fancy_label = r'A_\mathrm{K_S}'#~/~\mathrm{mag}$'
            elif label == 'logRHK':
                fancy_label = r'\log R_\mathrm{HK}^\prime'
            else:
                fancy_label = label

            print(fancy_label)
            ylabel = r'$\frac{\mathrm{d}F_\lambda}{\mathrm{d}'+fancy_label+r'}$'
        
        for ccd in [1,2,3,4]:
            ax = gs[index*4+ccd-1]
            in_ccd = (common_wavelength > (int(ccd)+3)*1000) & (common_wavelength < (int(ccd)+4)*1000)

            if index == 0:
                ydata = cannon_models['model_'+str(test)]['theta'][:,0]
            elif index == 1:
                ydata = np.sqrt(cannon_models['model_'+str(test)]['s2'][:])
            else:
                ydata = cannon_models['model_'+str(test)]['theta'][:,index-1]
                
            if test == 5:
                ydata_master = np.zeros(len(common_wavelength))
                ydata_master[~mask_common_wavelength] = ydata
                ydata = ydata_master
            elif test == 6:
                ydata_master = np.zeros(len(common_wavelength))
                ydata_master[mask_common_wavelength] = ydata
                ydata = ydata_master
            
            ax.plot(
                common_wavelength[in_ccd],
                ydata[in_ccd],
                lw = 0.5,
                c = color
            )
            y_minmax = np.min(ydata),np.max(ydata)

            # ax.set_ylim(1.1*y_minmax[0]-0.1*y_minmax[1],1.1*y_minmax[1]-0.1*y_minmax[0])
            ax.set_xlim(common_wavelength[in_ccd][0]-5,common_wavelength[in_ccd][-1]+5)

            if ccd == 1:
                ax.set_ylabel(ylabel,fontsize=labelfontsize)
            # else:
                # ax.set_yticks([])
            if index == rows-1:
                ax.set_xlabel(r'Wavelength $\lambda_\mathrm{air}~/~\mathrm{\AA}$')
            else:
                ax.set_xticks([])

        ax.set_ylim(-0.45,0.45)

    f.align_ylabels(gs)
    plt.tight_layout(w_pad=0,h_pad=0)
    plt.savefig('figures/cannon_linear_coefficients_model'+str(test)+'.pdf',dpi=200,bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
# Plot linear coefficients for all models in one figure, using different colormaps for the different versions

rows = 2 + 3+4+1+1

f, gs = plt.subplots(rows,4,figsize=(10,10))
gs = gs.flatten()

for test in [4,3,2,1]:

    if test == 1: cmap = 'Greys_r'; color = 'k'
    if test == 2: cmap = 'Blues_r'; color = 'C0'
    if test == 3: cmap = 'Oranges_r'; color = 'C1'
    if test == 4: cmap = 'Purples_r'; color = 'C4'

    for index in range(rows):

        for ccd in [1,2,3,4]:

            ax = gs[index*4+ccd-1]
            in_ccd = (common_wavelength > (int(ccd)+3)*1000) & (common_wavelength < (int(ccd)+4)*1000)

            # Median models
            if index == 0:
                ylabel = 'Median\n'+r'Model $F_\lambda$'
                labelfontsize = 10
                ydata = cannon_models['model_'+str(test)]['theta'][:,0]
                toplot = True

            # Scatter
            elif index == 1:
                ylabel = 'Model\nScatter'
                labelfontsize = 10
                ydata = np.sqrt(cannon_models['model_'+str(test)]['s2'])
                toplot = True

            else:
                if index == 2:
                    needed_label = 'teff'
                    fancy_label = r'T_\mathrm{eff}'#r'$T_\mathrm{eff}~/~\mathrm{K}$'
                elif index == 3:
                    needed_label = 'logg'
                    fancy_label = r'\log (g)'#~/~\mathrm{cm\,s^{-2}})$'
                elif index == 4:
                    needed_label = 'fe_h'
                    fancy_label = r'\mathrm{[Fe/H]}'
                elif index == 5:
                    needed_label = 'na_fe'
                    fancy_label = r'\mathrm{[Na/Fe]}'
                elif index == 6:
                    needed_label = 'mg_fe'
                    fancy_label = r'\mathrm{[Mg/Fe]}'
                elif index == 7:
                    needed_label = 'mn_fe'
                    fancy_label = r'\mathrm{[Mn/Fe]}'
                elif index == 8:
                    needed_label = 'ba_fe'
                    fancy_label = r'\mathrm{[Ba/Fe]}'
                elif index == 9:
                    needed_label = 'a_ks'
                    fancy_label = r'A_\mathrm{K_S}'
                elif index == 10:
                    needed_label = 'logRHK'
                    fancy_label = r'\log R_\mathrm{HK}^\prime'
                else:
                    raise ValueError('Index '+str(index)+' does not correspond to a known label.')

                if needed_label in cannon_models['model_'+str(test)]['label_names']:
                    label_index = cannon_models['model_'+str(test)]['label_names'].index(needed_label)
                    ydata = cannon_models['model_'+str(test)]['theta'][:,label_index+1]
                    ylabel = r'$\frac{\mathrm{d}F_\lambda}{\mathrm{d}'+fancy_label+r'}$'
                    labelfontsize = 15
                    toplot = True
                else:
                    toplot = False

            if toplot:
                ax.plot(
                    common_wavelength[in_ccd],
                    ydata[in_ccd],
                    lw = 0.3,
                    c = color
                )
                if ccd == 1:
                    ax.set_ylabel(ylabel,fontsize=labelfontsize)
                else:
                    ax.set_yticks([])
                if index == rows-1:
                    ax.set_xlabel(r'Wavelength $\lambda_\mathrm{air}~/~\mathrm{\AA}$')

                y_minmax = np.max([np.abs(np.min(ydata)), np.max(ydata)])
                ax.set_ylim(-1.1*y_minmax, 1.1*y_minmax)
                if index < 4:
                    ax.set_ylim(-0.3,0.3)

                ax.set_xlim(common_wavelength[in_ccd][0]-5,common_wavelength[in_ccd][-1]+5)

for i, ax in enumerate(gs):
    if i % 4 != 0:   # not in first column
        ax.tick_params(axis='y', labelleft=False)
    if i < len(gs) - 4:   # not in last row
        ax.tick_params(axis='x', labelbottom=False)

f.align_ylabels(gs)
plt.tight_layout(w_pad=0,h_pad=-0.35)
plt.savefig('figures/cannon_linear_coefficients_joint.pdf',dpi=200,bbox_inches='tight')
plt.show()
plt.close()

## 3.X Classification of Residuals

In [ ]:
# Crossmatch significant peaks with line lists by Hinkle+ 2000

if rerun_everything:
    # Hinkle+2000 data atomic line data from their Appendix I
    with open('data/hinkle_2000vnia.book.....H_atlas_appi', 'r') as f:
        lines = f.readlines()

    hinkle_lines_species = []
    hinkle_lines_wavelength = []

    for i in range(1,len(lines)):
        line = lines[i]
        line = line.replace('\n','')
        line = line.replace(' ','')
        if len(line) > 0:
            try:
                line = float(line.split(',')[0])
                hinkle_lines_species.append(species)
                hinkle_lines_wavelength.append(line)
            except:
                species = line

    hinkle_lines_table = Table()
    hinkle_lines_table['species'] = hinkle_lines_species
    hinkle_lines_table['wave_air'] = hinkle_lines_wavelength
    hinkle_lines_table.write('data/hinkle_2000vnia.book.....H_atlas_appi.fits', overwrite=True)

In [ ]:
significant_peaks_table = Table.read('data/significant_peaks_baseline_sample.fits')

hinkle_lines_table = Table.read('data/hinkle_2000_atlas/hinkle_2000_APPI_lines.txt', format = 'ascii', names=['species', 'wave_air'])
hinkle_spectra = Table.read('data/hinkle_2000_atlas/hinkle_2000_solar_arcturus_telluric_atlas.fits')

In [ ]:
# Create figures for inspecting Hinkle+2000 atlases of Sun and Arcuturus + significant line.

def set_line_window_ticks(ax, lambda0):
    candidate_offsets = [
        np.array([-0.4, -0.2, 0.0, 0.2, 0.4]),
        np.array([-0.3, -0.1, 0.1, 0.3, 0.5]),
        np.array([-0.2, 0.0, 0.2, 0.4, 0.6]),
    ]

    xmin, xmax = ax.get_xlim()

    for offsets in candidate_offsets:
        ticks = lambda0 + offsets
        if np.all((ticks > xmin) & (ticks < xmax)):
            ax.set_xticks(ticks)
            return ticks
        
for peak_index, significant_peak in enumerate(significant_peaks_table):

    window = 1.1

    hinkle_region = (hinkle_spectra['WAVELENGTH'] > significant_peak['wave']-window/2.0) & (hinkle_spectra['WAVELENGTH'] < significant_peak['wave']+window/2.0)
    hinkle_lines_in_region = (hinkle_lines_table['wave_air'] > significant_peak['wave']-window/2.1) & (hinkle_lines_table['wave_air'] < significant_peak['wave']+window/2.1)
    residual_region = (common_wavelength > significant_peak['wave']-window/2.0) & (common_wavelength < significant_peak['wave']+window/2.0)

    f, ax = plt.subplots(figsize=(5,4))

    l3, = ax.plot(
        hinkle_spectra['WAVELENGTH'][hinkle_region],
        hinkle_spectra['SOLARFLUX'][hinkle_region],
        c = 'k',
        lw = 0.5,
        label = 'Sun'
    )
    l4, = ax.plot(
        hinkle_spectra['WAVELENGTH'][hinkle_region],
        hinkle_spectra['ARCTURUS'][hinkle_region],
        c = 'k',
        ls = 'dashed',
        lw = 0.5,
        label = 'Arcturus'
    )
    l5, = ax.plot(
        hinkle_spectra['WAVELENGTH'][hinkle_region],
        hinkle_spectra['TELLURIC'][hinkle_region],
        c = 'darkblue',
        lw = 0.5,
        label = 'Telluric'
    )

    l1 = ax.fill_between(
        common_wavelength[residual_region],
        -0.2*np.ones(len(common_wavelength[residual_region])),
        np.log10(significance_residual_flux_percentiles[1][residual_region]),
        color = 'lightcoral',
        lw = 0.5,
        alpha = 0.2,
        label = r'$\log_{10}(\mathrm{Significance})$'
    )
    ax.plot(
        common_wavelength[residual_region],
        np.log10(significance_residual_flux_percentiles[1][residual_region]),
        c = 'lightcoral',
        lw = 0.5,
        label = '_nolegend_'
    )

    l2 = ax.axvline(significant_peak['wave'], color='red', lw=1, ls='--', ymin = 0.0, ymax=0.75, label = 'Significant Peak\nat '+r'$\lambda_\mathrm{air}=$'+"{:.2f}".format(significant_peak['wave'])+r'$\mathrm{\AA}$')
    
    leg1 = ax.legend(handles = [l1, l2], ncol=1, fontsize=8, loc='lower left', framealpha=0.9)
    ax.add_artist(leg1)
    leg2 = ax.legend(handles = [l3,l4, l5], ncol=1, fontsize=8, loc='lower right', framealpha=0.9)
    leg2.set_title('Hinkle+2000', prop={'size': 8})
    
    for hinkle_line in hinkle_lines_table[hinkle_lines_in_region]:
        ax.axvline(hinkle_line['wave_air'], color='grey', lw=1, alpha=0.7, ymin=0.8, ymax=0.87)
        ax.text(hinkle_line['wave_air'], 1.05, hinkle_line['species'], rotation=90, va='bottom', ha='center', fontsize=10, color='k')

    ax.set_xlabel(r'Wavelength $\lambda_\mathrm{air}~/~\mathrm{\AA}$', fontsize=12)
    ax.set_ylabel('Fluxes / a.u.', fontsize=12)
    ax.set_xlim(significant_peak['wave']-window/2.0, significant_peak['wave']+window/2.0)
    ax.set_ylim(-0.2,1.2)

    ax.set_title('GALAH DR4 Residual at '+r'$\lambda_\mathrm{air}=$'+"{:.2f}".format(significant_peak['wave'])+r'$\mathrm{\AA}$', fontsize=12, pad=10)

    plt.tight_layout()
    plt.savefig('significant_residual_regions/residual_region_'+"{:.0f}".format(significant_peak['wave'])+'.png', dpi=300, bbox_inches='tight')
    if peak_index < 2:  # Show the first few
        plt.show()
    plt.close()

In [ ]:
# Check for nearby lines in Hinkle+ 2000 within +-1 pixel of each significant peak, and classify the peaks accordingly
classification = dict()
classification['residual_peak_index'] = []
classification['residual_significance'] = []
classification['residual_wave_air'] = []
classification['ref_line_wave_air'] = []
classification['ref_line_distance_lambda'] = []
classification['ref_line_distance_pixels'] = []
classification['ref_line_source'] = []
classification['ref_line_species'] = []
classification['residual_classification'] = []
classification['classification_comments'] = []

significant_peaks_table['wave'] = np.round(significant_peaks_table['wave'],3)

for peak_index, significant_peak in enumerate(significant_peaks_table):

    classification['residual_peak_index'].append(peak_index)
    classification['residual_wave_air'].append(np.round(significant_peak['wave'],3))
    classification['residual_significance'].append(np.round(significant_peak['significance'],1))

    # within +-1 pixel, find the closest line in Hinkle+ 2000
    if (significant_peak['wave'] > 4719.0299) & (significant_peak['wave'] < 4893.8808):
        pixelscale = 0.0460
    elif (significant_peak['wave'] > 5655.5298) & (significant_peak['wave'] < 5863.0182):
        pixelscale = 0.0547
    elif (significant_peak['wave'] > 6486.0104) & (significant_peak['wave'] < 6726.4572):
        pixelscale = 0.0631
    elif (significant_peak['wave'] > 7689.1261) & (significant_peak['wave'] < 7874.3127):
        pixelscale = 0.0735

    window = 2 * pixelscale

    nearby_lines = hinkle_lines_table[np.abs(hinkle_lines_table['wave_air'] - significant_peak['wave']) < window]
    if len(nearby_lines) > 0:
        closest_line_index = np.argmin(np.abs(nearby_lines['wave_air'] - significant_peak['wave']))
        closest_line = nearby_lines[closest_line_index]

        classification['ref_line_species'].append(closest_line['species'])
        classification['ref_line_wave_air'].append(closest_line['wave_air'])
        classification['ref_line_distance_lambda'].append(np.round(np.abs(closest_line['wave_air'] - significant_peak['wave']),3))
        classification['ref_line_distance_pixels'].append(np.round(np.abs(closest_line['wave_air'] - significant_peak['wave'])/pixelscale,1))
        classification['ref_line_source'].append('Hinkle+2000')
        if closest_line['species'] == 'FeI':
            classification['residual_classification'].append('FeI Line')
        else:
            classification['residual_classification'].append('Other Line')
        classification['classification_comments'].append('Automatic classification')
    elif significant_peak['wave'] in [4861.492, 6563.119]:
        if significant_peak['wave'] == 4861.492:
            hinkle_species = 'HI'
            hinkle_line = 4861.32
            classification['residual_classification'].append(r'$\mathrm{H_\beta}$')
        elif significant_peak['wave'] == 6563.119:
            hinkle_species = 'HI'
            hinkle_line = 6562.819
            classification['residual_classification'].append(r'$\mathrm{H_\alpha}$')

        classification['ref_line_species'].append(hinkle_species)
        classification['ref_line_wave_air'].append(hinkle_line)
        classification['ref_line_distance_lambda'].append(np.round(np.abs(hinkle_line - significant_peak['wave']),3))
        classification['ref_line_distance_pixels'].append(np.round(np.abs(hinkle_line - significant_peak['wave'])/pixelscale,1))
        classification['ref_line_source'].append('Hinkle+2000')
        classification['classification_comments'].append('--')
    else:
        classification['ref_line_species'].append('None')
        classification['ref_line_wave_air'].append(np.nan)
        classification['ref_line_distance_lambda'].append(np.nan)
        classification['ref_line_distance_pixels'].append(np.nan)
        classification['ref_line_source'].append('--')
        classification['residual_classification'].append('--')
        classification['classification_comments'].append('--')

for key in ['residual_wave_air','ref_line_wave_air']:
    classification[key] = np.array(classification[key])

In [ ]:
reduction_sky_tellurics = Table.read('data/reduction_sky_tellurics.fits')

sky_line_peaks, sky_line_peaks_strength = find_peaks(reduction_sky_tellurics['sky'], height = 2)
sky_line_wave = reduction_sky_tellurics['wave'][sky_line_peaks]
sky_lines = Table()
sky_lines['wave'] = np.round(sky_line_wave, 3)
sky_lines['strength'] = np.round(sky_line_peaks_strength['peak_heights'],1)

sky_line_matches = np.array([
    # [6498.757, 6494.466], #?
    # [6522.419, 6522.104], #?
    # [6533.020, 6533.903], #?
    # [6544.062, ],
    [6553.654, 6553.275],
    [6568.798, 6568.987],
    [6577.316, 6577.632],
    # [6596.625, ],
    # [6604.007, ],
    # [6604.197, ],
    # [7712.279, ],
    [7712.646, 7712.720],
    [7715.072, 7714.263],
    # [7715.880, ],
    # [7716.909, ],
    [7718.085, 7719.041],
    [7723.671, 7722.569],
    [7726.023, 7727.42],
    [7749.396, 7748.294],
    [7750.646, 7751.087],
    [7759.980, 7759.245],
    # [7773.357, 7774.239], # <- actually OI triplet
    [7780.413, 7780.56],
    [7794.084, 7793.937],
    [7808.490, 7807.829],
    [7821.500, 7821.794],
    [7841.271, 7840.830],
    [7853.399, 7853.252],
    [7860.675, 7860.896],
    [7867.731, 7869.642],
    [7870.745, 7871.480]
])

for sky_line_match in sky_line_matches:
    index_in_classification = np.where(classification['residual_wave_air'] == sky_line_match[1])[0]
    if len(index_in_classification) > 0:
        index_in_classification = index_in_classification[0]
        classification['ref_line_species'][index_in_classification] = 'Sky'
        classification['ref_line_wave_air'][index_in_classification] = sky_line_match[0]
        classification['ref_line_distance_lambda'][index_in_classification] = np.round(np.abs(sky_line_match[0] - sky_line_match[1]),3)
        classification['ref_line_distance_pixels'][index_in_classification] = np.round(np.abs(sky_line_match[0] - sky_line_match[1])/pixelscale,1)
        classification['ref_line_source'][index_in_classification] = 'Sky Obs.'
        classification['residual_classification'][index_in_classification] = 'Sky Line'

        find_strength = sky_lines['strength'][sky_lines['wave'] == sky_line_match[0]]

        classification['classification_comments'][index_in_classification] = 'Sky line strength '+str(find_strength[0])

In [ ]:
# Guided identifications via comparison of Hinkle+2000 & Linelist

manual_classifications = np.array([
    # lambda , lambda   , source   , LOGG, SYN_, species  , Classif., comments
    # Resid.   DR4      ,            FLAG, FLAG
    # > 5 Sigma
    [4748.148, 4748.1760, 'DR4/K07', '?', '-', 'MnI'    , 'Manual'            , r'Visible but unidentified in Hinkle+2000'],
    [4771.47 , 4771.4503, 'DR4/K09', '?', '-', 'ScI/CrI', 'Manual'            , r'Visible but unidentified in Hinkle+2000'],
    [5693.656, 5693.6162, 'DR4/K07', 'N', 'U', 'FeI'    , 'FeI Line'          , r'LOGF\_FLAG=N, SYN\_FLAG=U in linelist'],
    [5754.209, 5754.4025, 'DR4/K07', '?', 'N', 'FeI'    , 'FeI Line'          , r'SYN\_FLAG=N in linelist'], # <- SiI is mismatch with automatic comparison to Hinkle+2000
    [5667.592, 5667.5940, 'DR4/K10', '-', '-', 'TiI'    , 'Fake Synthesis'    , r'Wrong $\log gf$. No TiI here (actually FeI)'],
    [7799.964, 7799.9957, 'DR4/K07', '-', '-', 'SiI'    , 'Missing Synthesis' , r'Too low $\log gf$'],
    # 4 < Sigma < 5
    [5831.062, 5831.0567, 'DR4/K07', '-', '-', 'FeI'    , 'Fake Synthesis'    , r'Wrong $\log gf$. No FeI here (no line visible in Hinkle+2000)'],
    # Unidentified
    [7732.344, np.nan,    '?'      , '-', '-', '?'      , 'Unidentified'      , r'Unidentified shallow, broad stellar feature (correlates with [Fe/H] in M3, visible in Hinkle+2000).'],  
])

for manual_classification in manual_classifications:
    index_in_classification = np.where(classification['residual_wave_air'] == float(manual_classification[0]))[0]
    if len(index_in_classification) > 0:
        index_in_classification = index_in_classification[0]

        classification['ref_line_species'][index_in_classification] = manual_classification[5]
        classification['ref_line_wave_air'][index_in_classification] = float(manual_classification[1])
        classification['ref_line_distance_lambda'][index_in_classification] = np.round(np.abs(float(manual_classification[1]) - classification['residual_wave_air'][index_in_classification]),3)
        classification['ref_line_distance_pixels'][index_in_classification] = np.round(np.abs(float(manual_classification[1]) - classification['residual_wave_air'][index_in_classification])/pixelscale,1)
        classification['ref_line_source'][index_in_classification] = manual_classification[2]
        classification['residual_classification'][index_in_classification] = manual_classification[6]
        classification['classification_comments'][index_in_classification] = manual_classification[7]

In [ ]:
classified = Table(classification)
classified.write('data/significant_peaks_classification.fits', overwrite=True)
classified.sort('residual_significance', reverse=True)
classified[:10]

In [ ]:
unique_classifications, unique_classifications_indices, unique_classifications_counts = np.unique(classified['residual_classification'], return_counts=True, return_index=True)

unique_classifications_sorted = classified[unique_classifications_indices]
unique_classifications_sorted.sort('residual_significance', reverse=True)

# Create LaTeX table based on classified one.
latex_table = r'''\begin{tabular}{cccccccccp{4.75cm}}
\hline
Lines in & Classification & Largest & Residual                            & Reference                           &                  &            & Reference & Reference & Comments \\
Class.   &                & Signif. & $\lambda_\mathrm{air}~/~\text{\AA}$ & $\lambda_\mathrm{air}~/~\text{\AA}$ & $\Delta \lambda$ & $\Delta$px & Source    & Species   &          \\
\hline
'''
for row in unique_classifications_sorted:
    latex_table += f"{unique_classifications_counts[row['residual_classification'] == unique_classifications][0]} & {row['residual_classification']} & {row['residual_significance']:.1f} & {row['residual_wave_air']:.3f} & "
    if np.isnan(row['ref_line_wave_air']):
        latex_table += "-- & "
    else:
        latex_table += f"{row['ref_line_wave_air']:.3f} & "
    if np.isnan(row['ref_line_distance_lambda']):
        latex_table += "-- & -- & "
    else:
        latex_table += f"{row['ref_line_distance_lambda']:.3f} & {row['ref_line_distance_pixels']:.1f} & "
    latex_table += f"{row['ref_line_source']} & "
    latex_table += f"{row['ref_line_species']} & {row['classification_comments']} \\\\\n"
latex_table += r'''\dots & \dots & \dots & \dots & \dots & \dots & \dots & \dots & \dots & \dots \\
\hline
\end{tabular}'''
np.savetxt('tex_text/tabular_significant_peaks_classification.tex', [latex_table], fmt='%s')
print(latex_table)

## 3.Y Activity Indicators

In [ ]:
# https://ui.adsabs.harvard.edu/abs/2018AJ....156..180W
Wise2018_lines = [
4827.46,
4861.33,
5707.00,
6562.81
]

# https://ui.adsabs.harvard.edu/abs/2019MNRAS.485.4804L
Lisogorskyi2019_lines = Table.read('data/Lisogorskyi_2019_MNRAS_485_4804_TableA1.fits')
Lisogorskyi2019_lines = Lisogorskyi2019_lines[(
    ((Lisogorskyi2019_lines['center'] >= 4719.0299) & (Lisogorskyi2019_lines['center'] <= 4893.8759)) |
    ((Lisogorskyi2019_lines['center'] >= 5655.5298) & (Lisogorskyi2019_lines['center'] <= 5863.0069)) |
    ((Lisogorskyi2019_lines['center'] >= 6486.0104) & (Lisogorskyi2019_lines['center'] <= 6726.4214)) |
    ((Lisogorskyi2019_lines['center'] >= 7689.1261) & (Lisogorskyi2019_lines['center'] <= 7874.2726))
)]
plt.scatter(
    Lisogorskyi2019_lines['center'],
    Lisogorskyi2019_lines['logS_ew_p'],
)
plt.show()
plt.close()
np.array(Lisogorskyi2019_lines['center'])

In [ ]:
residuals = cannon_models['model_4']['theta'][:,0]

large_residuals = np.where(np.abs(residuals) > 0.075)[0]

count = 0

significant_residuals = []

for residual_index in np.arange(len(large_residuals)-1):
    this_one = common_wavelength[large_residuals][residual_index]
    next_one = common_wavelength[large_residuals][residual_index+1]

    if next_one - this_one > 0.1:
        print(count, this_one, next_one, next_one - this_one)
        significant_residuals.append(this_one)
        count += 1

significant_residuals = np.array(significant_residuals)

In [ ]:
significant_residuals

In [ ]:
residuals = cannon_models['model_4']['theta'][:,5]

large_residuals = np.where(np.abs(residuals) > 0.075)[0]

count = 0

significant_residuals = []

for residual_index in np.arange(len(large_residuals)-1):
    this_one = common_wavelength[large_residuals][residual_index]
    next_one = common_wavelength[large_residuals][residual_index+1]

    if next_one - this_one > 0.1:
        print(count, this_one, next_one, next_one - this_one)
        significant_residuals.append(this_one)
        count += 1

significant_residuals = np.array(significant_residuals)

In [ ]:
investigate = Table()
investigate['lambda'] = common_wavelength
investigate['flux'] = cannon_models['model_4']['theta'][:,0]
investigate['dflux_dteff'] = cannon_models['model_4']['theta'][:,1]
investigate['dflux_dlogg'] = cannon_models['model_4']['theta'][:,2]
investigate['dflux_dfeh'] = cannon_models['model_4']['theta'][:,3]
investigate['dflux_daks'] = cannon_models['model_4']['theta'][:,4]
investigate['dflux_dlogrhk'] = cannon_models['model_4']['theta'][:,5]
investigate.write('data/model_coefficients_test_4.fits',overwrite=True)

In [ ]:
for index, line in enumerate(np.array(Lisogorskyi2019_lines['center'])):

    in_wavelength = (common_wavelength > line - 2) & (common_wavelength < line + 2)

    print(Lisogorskyi2019_lines[index])
    plt.figure(figsize=(10,3))
    plt.plot(
        common_wavelength[in_wavelength],
        cannon_models['model_4']['theta'][in_wavelength,0],
        lw = 0.5
    )
    plt.plot(
        common_wavelength[in_wavelength],
        cannon_models['model_4']['theta'][in_wavelength,5],
        lw = 0.5
    )
    plt.ylim(-0.2,0.2)
    plt.axvline(line)
    plt.show()
    plt.close()

In [ ]:
def plot_fit(star_index, test = 3):

    f, gs = plt.subplots(4,1,figsize=(10,8),sharey=True)
    for ccd in [1,2,3,4]:
        in_ccd = (common_wavelength > (int(ccd)+3)*1000) & (common_wavelength < (int(ccd)+4)*1000)
        ax = gs[ccd-1]
        ax.plot(
            common_wavelength[in_ccd],
            cannon_models['model_'+str(test)]['flux'][star_index,in_ccd],
            c = 'grey', label = 'Residual Flux DR4', lw=0.5
        )
        ax.plot(
            common_wavelength[in_ccd],
            cannon_models['model_'+str(test)]['flux_out'][star_index,in_ccd],
            c = 'C0', label = 'Fit to Residual Flux', lw=0.5
        )
        ax.plot(
            common_wavelength[in_ccd],
            cannon_models['model_'+str(test)]['flux'][star_index,in_ccd] - cannon_models['model_'+str(test)]['flux_out'][star_index,in_ccd] + 0.25,
            c = 'C3', label = 'Residual of Residual + 0.25', lw=0.5
        )
        ax.legend(ncol=3)
    ax.set_ylim(-0.25,0.5)
    f.suptitle(str(star_index)+' '+str(cannon_models['model_'+str(test)]['sobject_id'][star_index])+': TEFF '+str(np.round(cannon_models['model_'+str(test)]['labels'][star_index, 0]))+', log(g) '+str(np.round(cannon_models['model_'+str(test)]['labels'][star_index, 1],2))+', [Fe/H] '+str(np.round(cannon_models['model_'+str(test)]['labels'][star_index, 2],2)))

    plt.tight_layout()
    
    plt.savefig('figures/residuals_test_'+str(test)+'_'+str(cannon_models['model_'+str(test)]['sobject_id'][star_index])+'.pdf',bbox_inches='tight',dpi=200)
    plt.show()
    plt.close()
    
to_plot = [
    0, # first star -> 
    100, # missed lines in CCD1; emission lines in CCD4
    200, # DIBs & KI
]

sobject_ids = [
    # # 170711005801103, # Halpha and Hbeta as well as line at 7800 and 5755
    # # 170711005801334, # missed lines in CCD1; emission lines in CCD4
    # # 170712003101121, # DIBs & KI,
    # # 170710003201320, # highest logRHK
    # 170711003001093,
    # 170711003001103, # high E(B-V) and A(Ks)
    # 170711003001151, # high E(B-V) and A(Ks)
    170711001501201, # cool giant recovered as hot dwarf
]

for sobject_id in sobject_ids:
    for test in tests_run:
        match = np.where(cannon_models['model_'+str(test)]['sobject_id'] == sobject_id)[0]
        if len(match) > 0:
            print(test, sobject_id)
            plot_fit(match[0], test)